# Notebook 09 — XAI Validation and Analyst Evidence

**Praxis:** *Explainable AI for Low-Altitude Aircraft Threat Detection in Cyber-Physical Airspace Surveillance*

This notebook validates the explainability layer produced by Notebooks 04–07. It does **not** retrain the primary models and it does **not** alter H1–H3. Instead, it evaluates the supporting XAI objective:

> **How faithfully, consistently, traceably, and efficiently do feature-, temporal-, and relation-level explanations characterize the evidence contributing to candidate low-altitude event risk scores?**

The notebook performs six XAI checks:

1. **Global explanation synthesis** across the Transformer and R-GCN.
2. **Perturbation fidelity** by removing top-ranked evidence and comparing the effect with random evidence removal.
3. **Explanation stability** under small transformed-input perturbations.
4. **Relation consistency** by comparing R-GCN relation diagnostics with relation-removal performance effects.
5. **Provenance and traceability** from analyst evidence back to event and flow records.
6. **Analyst-facing case studies and explanation latency** for representative held-out events.

### Interpretation boundary

The aircraft and NF-UNSW-NB15-v3 sources remain independent historical datasets connected through the controlled replay procedure established in Notebook 03. XAI outputs describe **model influence**, not causality, site coincidence, or proof that an aircraft generated a network flow or attack.


> Corrected build: Pylance bool-like audit correction for pandas/NumPy boolean scalars.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import sys
import time
import warnings
from dataclasses import dataclass
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence, cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

RANDOM_STATE = 42
MODEL_VERSION = "xai_validation_v2"
PRIMARY_TRANSFORMER_FEATURE_SET = "fused"
PRIMARY_RGCN_VARIANT = "full_lag_relation"
ABLATION_RGCN_VARIANT = "lag_relation_ablation"

FAST_MODE = os.getenv("XAI_FAST_MODE", "0").strip() == "1"
TOP_K = int(os.getenv("XAI_TOP_K", "10"))
FIDELITY_RANDOM_REPEATS = int(os.getenv("XAI_FIDELITY_RANDOM_REPEATS", "2" if FAST_MODE else "5"))
STABILITY_REPEATS = int(os.getenv("XAI_STABILITY_REPEATS", "1" if FAST_MODE else "3"))
NOISE_STD = float(os.getenv("XAI_NOISE_STD", "0.01"))
TRANSFORMER_FIDELITY_SAMPLE_MAX = int(os.getenv("XAI_TRANSFORMER_FIDELITY_SAMPLE_MAX", "48" if FAST_MODE else "128"))
TRANSFORMER_STABILITY_SAMPLE_MAX = int(os.getenv("XAI_TRANSFORMER_STABILITY_SAMPLE_MAX", "32" if FAST_MODE else "96"))
RGCN_ATTRIBUTION_SAMPLE_MAX = int(os.getenv("XAI_RGCN_ATTRIBUTION_SAMPLE_MAX", "32" if FAST_MODE else "96"))
CASE_STUDY_COUNT = int(os.getenv("XAI_CASE_STUDY_COUNT", "3" if FAST_MODE else "5"))
LATENCY_SAMPLE_MAX = int(os.getenv("XAI_LATENCY_SAMPLE_MAX", "3" if FAST_MODE else "10"))
LOCAL_HOPS = int(os.getenv("XAI_LOCAL_HOPS", "2"))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = Path.cwd()
BASE_DIR = PROJECT_ROOT / "data" / "processed" / "point_event_v2"
ALIGNMENT_DIR = BASE_DIR / "nf3_alignment_v2"
MODELING_DIR = BASE_DIR / "modeling_v4"
TRANSFORMER_DIR = BASE_DIR / "transformer_v2"
RGCN_DIR = BASE_DIR / "rgcn_leakage_safe_v2"
RESULTS_DIR = BASE_DIR / "results_v2"
OUTPUT_DIR = BASE_DIR / "xai_leakage_safe_v2"
FIGURES_DIR = OUTPUT_DIR / "figures"
ANALYST_CARDS_DIR = OUTPUT_DIR / "analyst_cards"

FEATURES_IN = ALIGNMENT_DIR / "candidate_point_events_v2_with_context_and_nf3_primary.csv"
SPLITS_IN = MODELING_DIR / "model_split_assignments_v4.csv"
FEATURE_MANIFEST_IN = MODELING_DIR / "feature_manifest_v4.json"
TARGET_DEFINITION_IN = MODELING_DIR / "model_target_definition_v4.json"
BASELINE_PERMUTATION_IN = MODELING_DIR / "permutation_importance_v4.csv"

TRANSFORMER_MODEL_IN = TRANSFORMER_DIR / "models" / "fused_temporal_transformer.pt"
TRANSFORMER_PREDICTIONS_IN = TRANSFORMER_DIR / "temporal_transformer_predictions_v2.csv"
TRANSFORMER_ATTRIBUTION_IN = TRANSFORMER_DIR / "temporal_transformer_feature_attribution_v2.csv"
TRANSFORMER_ATTENTION_IN = TRANSFORMER_DIR / "temporal_transformer_attention_summary_v2.csv"

RGCN_MODEL_IN = RGCN_DIR / "models" / "full_lag_relation_rgcn.pt"
RGCN_PREDICTIONS_IN = RGCN_DIR / "rgcn_predictions_v2.csv"
RGCN_ATTRIBUTION_IN = RGCN_DIR / "rgcn_feature_attribution_v2.csv"
RGCN_RELATION_NORMS_IN = RGCN_DIR / "rgcn_relation_weight_norms_v2.csv"
RGCN_SELECTED_EDGES_IN = RGCN_DIR / "selected_event_flow_edges_v2.csv"
RGCN_NODE_MANIFEST_IN = RGCN_DIR / "rgcn_node_manifest_v2.csv"
RGCN_GRAPH_MANIFEST_IN = RGCN_DIR / "rgcn_graph_manifest_v2.json"
RGCN_CONFIG_IN = RGCN_DIR / "rgcn_config_v2.json"
RGCN_RESULTS_IN = RGCN_DIR / "rgcn_results_v2.csv"

GLOBAL_SUMMARY_OUT = OUTPUT_DIR / "xai_global_summary_v2.csv"
FIDELITY_OUT = OUTPUT_DIR / "xai_fidelity_results_v2.csv"
STABILITY_OUT = OUTPUT_DIR / "xai_stability_results_v2.csv"
RELATION_CONSISTENCY_OUT = OUTPUT_DIR / "xai_relation_consistency_v2.csv"
LATENCY_OUT = OUTPUT_DIR / "xai_explanation_latency_v2.csv"
LATENCY_SAMPLES_OUT = OUTPUT_DIR / "xai_explanation_latency_samples_v2.csv"
PROVENANCE_OUT = OUTPUT_DIR / "xai_provenance_audit_v2.csv"
MODEL_MATRIX_OUT = OUTPUT_DIR / "xai_model_comparison_matrix_v2.csv"
CASE_STUDIES_OUT = OUTPUT_DIR / "xai_case_studies_v2.csv"
REFERENCE_OUT = OUTPUT_DIR / "xai_reference_v2.json"
CONFIG_OUT = OUTPUT_DIR / "xai_config_v2.json"
SUMMARY_OUT = OUTPUT_DIR / "xai_build_summary_v2.json"
NARRATIVE_OUT = OUTPUT_DIR / "xai_results_narrative_v2.md"
DATA_DICTIONARY_OUT = OUTPUT_DIR / "xai_data_dictionary_v2.csv"
CHECKSUMS_OUT = OUTPUT_DIR / "SHA256SUMS.txt"

for directory in [OUTPUT_DIR, FIGURES_DIR, ANALYST_CARDS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"XAI output:   {OUTPUT_DIR}")
print(f"Device:       {DEVICE}")
print(f"Fast mode:    {FAST_MODE}")


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def require_int(value: object, field_name: str) -> int:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")
    if isinstance(value, bool):
        return int(value)
    text = value.decode("utf-8", errors="strict") if isinstance(value, bytes) else str(value)
    text = text.strip()
    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{field_name} cannot be missing.")
    numeric = float(text)
    if not math.isfinite(numeric) or not numeric.is_integer():
        raise ValueError(f"{field_name} must be a finite whole number; received {value!r}.")
    return int(numeric)


def require_float(value: object, field_name: str) -> float:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")
    text = value.decode("utf-8", errors="strict") if isinstance(value, bytes) else str(value)
    text = text.strip()
    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{field_name} cannot be missing.")
    numeric = float(text)
    if not math.isfinite(numeric):
        raise ValueError(f"{field_name} must be finite; received {value!r}.")
    return numeric


def require_text(value: object, field_name: str) -> str:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")
    text = value.decode("utf-8", errors="strict") if isinstance(value, bytes) else str(value)
    text = text.strip()
    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{field_name} cannot be empty or missing.")
    return text


def require_utc_timestamp(value: object, field_name: str) -> pd.Timestamp:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")
    if isinstance(value, pd.Timestamp):
        timestamp = value
    elif isinstance(value, (datetime, date, np.datetime64, str, int, float)):
        timestamp = pd.Timestamp(value)
    elif isinstance(value, bytes):
        timestamp = pd.Timestamp(value.decode("utf-8", errors="strict"))
    else:
        timestamp = pd.Timestamp(str(value))
    if pd.isna(timestamp):
        raise ValueError(f"{field_name} cannot be missing.")
    if timestamp.tzinfo is None:
        timestamp = timestamp.tz_localize("UTC")
    else:
        timestamp = timestamp.tz_convert("UTC")
    return timestamp


def safe_torch_load(path: Path) -> dict[str, Any]:
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    if not isinstance(payload, dict):
        raise TypeError(f"Expected a dictionary checkpoint at {path}.")
    return cast(dict[str, Any], payload)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def make_dense_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=2)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(numeric_columns: list[str], categorical_columns: list[str]) -> ColumnTransformer:
    transformers: list[tuple[str, Pipeline, list[str]]] = []
    if numeric_columns:
        transformers.append(
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_columns,
            )
        )
    if categorical_columns:
        transformers.append(
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_dense_one_hot_encoder()),
                    ]
                ),
                categorical_columns,
            )
        )
    if not transformers:
        raise ValueError("No usable predictors remain for preprocessing.")
    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)


def transformed_feature_names(preprocessor: ColumnTransformer) -> list[str]:
    try:
        names_obj = preprocessor.get_feature_names_out()
        return [str(value) for value in np.asarray(names_obj, dtype=object).tolist()]
    except (AttributeError, ValueError):
        names: list[str] = []
        for transformer_name, transformer, columns in preprocessor.transformers_:
            if transformer_name == "remainder" or transformer == "drop":
                continue
            column_names = [str(value) for value in list(columns)]
            if isinstance(transformer, Pipeline) and "onehot" in transformer.named_steps:
                encoder = transformer.named_steps["onehot"]
                try:
                    encoded_names = encoder.get_feature_names_out(column_names)
                    names.extend([str(value) for value in np.asarray(encoded_names, dtype=object).tolist()])
                except AttributeError:
                    names.extend(column_names)
            else:
                names.extend(column_names)
        return names


def as_float32_matrix(value: object, expected_rows: int, name: str) -> np.ndarray:
    matrix = np.asarray(value, dtype=np.float32)
    if matrix.ndim != 2 or matrix.shape[0] != expected_rows:
        raise ValueError(f"Unexpected {name} matrix shape: {matrix.shape}.")
    if not np.isfinite(matrix).all():
        raise ValueError(f"{name} matrix contains non-finite values.")
    return matrix


def safe_auprc(y_true: np.ndarray, probabilities: np.ndarray) -> float:
    labels = np.asarray(y_true, dtype=np.int8)
    probs = np.asarray(probabilities, dtype=float)
    if np.unique(labels).size < 2:
        return float("nan")
    return float(average_precision_score(labels, probs))


def balanced_positions(labels: np.ndarray, maximum_rows: int, seed: int) -> np.ndarray:
    values = np.asarray(labels, dtype=np.int8)
    positions = np.arange(len(values), dtype=np.int64)
    if len(positions) <= maximum_rows:
        return positions
    rng = np.random.default_rng(seed)
    positive = positions[values == 1]
    negative = positions[values == 0]
    positive_take = min(len(positive), maximum_rows // 2)
    selected_positive = rng.choice(positive, size=positive_take, replace=False) if positive_take else np.empty(0, dtype=np.int64)
    remaining = maximum_rows - len(selected_positive)
    negative_take = min(len(negative), remaining)
    selected_negative = rng.choice(negative, size=negative_take, replace=False) if negative_take else np.empty(0, dtype=np.int64)
    selected = np.concatenate([selected_positive, selected_negative])
    if len(selected) < maximum_rows:
        pool = np.setdiff1d(positions, selected, assume_unique=False)
        add_count = min(len(pool), maximum_rows - len(selected))
        if add_count:
            selected = np.concatenate([selected, rng.choice(pool, size=add_count, replace=False)])
    return np.sort(selected.astype(np.int64, copy=False))


def top_k_jaccard(reference: np.ndarray, candidate: np.ndarray, k: int) -> float:
    ref = np.asarray(reference, dtype=float)
    cand = np.asarray(candidate, dtype=float)
    k_eff = max(1, min(k, len(ref), len(cand)))
    ref_set = set(np.argsort(-ref)[:k_eff].tolist())
    cand_set = set(np.argsort(-cand)[:k_eff].tolist())
    union = ref_set | cand_set
    return float(len(ref_set & cand_set) / len(union)) if union else 1.0


def spearman_rank_correlation(a: np.ndarray, b: np.ndarray) -> float:
    first = pd.Series(np.asarray(a, dtype=float)).rank(method="average").to_numpy(dtype=float)
    second = pd.Series(np.asarray(b, dtype=float)).rank(method="average").to_numpy(dtype=float)
    if len(first) < 2 or float(np.std(first)) == 0.0 or float(np.std(second)) == 0.0:
        return float("nan")
    return float(np.corrcoef(first, second)[0, 1])


def synchronize_device() -> None:
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


seed_everything(RANDOM_STATE)


## 1. Load and validate frozen upstream artifacts

The notebook uses the **frozen checkpoints and split assignments** from Notebooks 04–07. The reconstructed feature matrices are checked against the saved model dimensions and Transformer feature-name manifest before any perturbation test is accepted.


In [ ]:
required_paths = [
    FEATURES_IN,
    SPLITS_IN,
    FEATURE_MANIFEST_IN,
    TARGET_DEFINITION_IN,
    TRANSFORMER_MODEL_IN,
    TRANSFORMER_PREDICTIONS_IN,
    TRANSFORMER_ATTRIBUTION_IN,
    TRANSFORMER_ATTENTION_IN,
    RGCN_MODEL_IN,
    RGCN_PREDICTIONS_IN,
    RGCN_ATTRIBUTION_IN,
    RGCN_RELATION_NORMS_IN,
    RGCN_SELECTED_EDGES_IN,
    RGCN_NODE_MANIFEST_IN,
    RGCN_GRAPH_MANIFEST_IN,
    RGCN_CONFIG_IN,
    RGCN_RESULTS_IN,
]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Notebook 09 requires completed Notebooks 03–07. Missing artifacts:\n"
        + "\n".join(missing_paths)
    )

features = pd.read_csv(FEATURES_IN, low_memory=False)
splits = pd.read_csv(SPLITS_IN, low_memory=False)
feature_manifest = json.loads(FEATURE_MANIFEST_IN.read_text(encoding="utf-8"))
target_definition = json.loads(TARGET_DEFINITION_IN.read_text(encoding="utf-8"))
transformer_checkpoint = safe_torch_load(TRANSFORMER_MODEL_IN)
rgcn_checkpoint = safe_torch_load(RGCN_MODEL_IN)
rgcn_graph_manifest = json.loads(RGCN_GRAPH_MANIFEST_IN.read_text(encoding="utf-8"))
rgcn_config = json.loads(RGCN_CONFIG_IN.read_text(encoding="utf-8"))

transformer_predictions_saved = pd.read_csv(TRANSFORMER_PREDICTIONS_IN, low_memory=False)
transformer_attribution_saved = pd.read_csv(TRANSFORMER_ATTRIBUTION_IN, low_memory=False)
transformer_attention_saved = pd.read_csv(TRANSFORMER_ATTENTION_IN, low_memory=False)
rgcn_predictions_saved = pd.read_csv(RGCN_PREDICTIONS_IN, low_memory=False)
rgcn_attribution_saved = pd.read_csv(RGCN_ATTRIBUTION_IN, low_memory=False)
rgcn_relation_norms_saved = pd.read_csv(RGCN_RELATION_NORMS_IN, low_memory=False)
selected_edges = pd.read_csv(RGCN_SELECTED_EDGES_IN, low_memory=False)
node_manifest = pd.read_csv(RGCN_NODE_MANIFEST_IN, low_memory=False)
rgcn_results_saved = pd.read_csv(RGCN_RESULTS_IN, low_memory=False)

ID_COLUMN = "event_id"
TIME_COLUMN = "aircraft_event_time_utc"
GROUP_COLUMN = "aircraft_key"
EPISODE_COLUMN = "episode_id"
TARGET_COLUMN = "model_target"
SPLIT_COLUMN = "chronological_split"

split_columns = [ID_COLUMN, TARGET_COLUMN, SPLIT_COLUMN]
for optional_column in ["cyber_attack_window_label", "aircraft_group_split"]:
    if optional_column in splits.columns:
        split_columns.append(optional_column)

model_data = features.merge(
    splits[split_columns],
    on=ID_COLUMN,
    how="inner",
    validate="one_to_one",
)
if len(model_data) != len(features) or len(model_data) != len(splits):
    raise ValueError(
        "Feature and split rows do not match one-to-one: "
        f"features={len(features):,}, splits={len(splits):,}, merged={len(model_data):,}."
    )

model_data[ID_COLUMN] = model_data[ID_COLUMN].astype("string")
model_data[GROUP_COLUMN] = model_data[GROUP_COLUMN].astype("string")
model_data[TIME_COLUMN] = model_data[TIME_COLUMN].map(
    lambda value: require_utc_timestamp(value, TIME_COLUMN)
)
model_data[TARGET_COLUMN] = pd.to_numeric(model_data[TARGET_COLUMN], errors="raise").astype("int8")
model_data[SPLIT_COLUMN] = model_data[SPLIT_COLUMN].astype("string")
if EPISODE_COLUMN in model_data.columns:
    model_data[EPISODE_COLUMN] = model_data[EPISODE_COLUMN].astype("string")
model_data = model_data.sort_values([TIME_COLUMN, ID_COLUMN], kind="mergesort").reset_index(drop=True)

split_order = model_data[SPLIT_COLUMN].map({"train": 0, "validation": 1, "test": 2})
if split_order.isna().any() or not split_order.is_monotonic_increasing:
    raise ValueError("Chronological split assignments are not monotonic after event-time sorting.")

print(f"Point events: {len(model_data):,}")
print(f"Test events:  {int(model_data[SPLIT_COLUMN].eq('test').sum()):,}")
print(f"Targets:      {model_data[TARGET_COLUMN].value_counts().sort_index().to_dict()}")
print(f"Transformer checkpoint: {TRANSFORMER_MODEL_IN.name}")
print(f"R-GCN checkpoint:       {RGCN_MODEL_IN.name}")


## 2. Reconstruct the fused event matrix and temporal Transformer

The feature preprocessor is deterministically rebuilt from the same training partition and raw feature list used by Notebook 05. The transformed feature names must exactly match the feature names stored in the Transformer checkpoint.


In [ ]:
raw_numeric_columns = [str(value) for value in transformer_checkpoint.get("raw_numeric_columns", [])]
raw_categorical_columns = [str(value) for value in transformer_checkpoint.get("raw_categorical_columns", [])]
raw_columns = raw_numeric_columns + raw_categorical_columns
missing_raw_columns = sorted(set(raw_columns) - set(model_data.columns))
if missing_raw_columns:
    raise KeyError(f"Required Transformer raw features are missing: {missing_raw_columns}")

for column in raw_numeric_columns:
    model_data[column] = pd.to_numeric(model_data[column], errors="coerce")
for column in raw_categorical_columns:
    model_data[column] = model_data[column].astype("string")

train_mask = model_data[SPLIT_COLUMN].eq("train")
fused_preprocessor = build_preprocessor(raw_numeric_columns, raw_categorical_columns)
fused_preprocessor.fit(model_data.loc[train_mask, raw_columns])
fused_matrix = as_float32_matrix(
    fused_preprocessor.transform(model_data[raw_columns]),
    len(model_data),
    "fused event feature",
)
fused_feature_names = transformed_feature_names(fused_preprocessor)
checkpoint_feature_names = [str(value) for value in transformer_checkpoint.get("transformed_feature_names", [])]

if fused_feature_names != checkpoint_feature_names:
    mismatch = [
        (index, observed, expected)
        for index, (observed, expected) in enumerate(zip(fused_feature_names, checkpoint_feature_names))
        if observed != expected
    ]
    raise ValueError(
        "Reconstructed Transformer feature names do not match the frozen checkpoint. "
        f"Observed={len(fused_feature_names)}, expected={len(checkpoint_feature_names)}, "
        f"first_mismatches={mismatch[:5]}"
    )

architecture = cast(dict[str, Any], transformer_checkpoint["architecture"])
if fused_matrix.shape[1] != require_int(architecture["input_dim"], "Transformer input_dim"):
    raise ValueError("Reconstructed fused feature dimension does not match Transformer checkpoint.")

TIME_GAP_MEAN = require_float(transformer_checkpoint["time_gap_mean_log1p"], "time-gap mean")
TIME_GAP_STD = require_float(transformer_checkpoint["time_gap_std_log1p"], "time-gap std")
SEQUENCE_LENGTH = require_int(architecture["sequence_length"], "Transformer sequence length")

event_times = model_data[TIME_COLUMN]
site_gap_seconds = event_times.diff().dt.total_seconds().fillna(0.0).clip(lower=0.0)
all_gap_log = np.log1p(site_gap_seconds.to_numpy(dtype=np.float64))
standardized_time_gap = ((all_gap_log - TIME_GAP_MEAN) / TIME_GAP_STD).astype(np.float32)


class TemporalTransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, feedforward_dim: int, dropout: float) -> None:
        super().__init__()
        self.norm_attention = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.dropout_attention = nn.Dropout(dropout)
        self.norm_feedforward = nn.LayerNorm(d_model)
        self.feedforward = nn.Sequential(
            nn.Linear(d_model, feedforward_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feedforward_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(
        self,
        inputs: torch.Tensor,
        causal_mask: torch.Tensor,
        padding_mask: torch.Tensor,
        return_attention: bool = False,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        normalized = self.norm_attention(inputs)
        attention_output, attention_weights = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=causal_mask,
            key_padding_mask=padding_mask,
            need_weights=return_attention,
            average_attn_weights=False,
        )
        hidden = inputs + self.dropout_attention(attention_output)
        hidden = hidden + self.feedforward(self.norm_feedforward(hidden))
        return hidden, attention_weights if return_attention else None


class TemporalTransformerClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int,
        sequence_length: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        feedforward_dim: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.sequence_length = int(sequence_length)
        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.time_projection = nn.Sequential(nn.Linear(1, d_model), nn.Tanh())
        self.position_embedding = nn.Embedding(sequence_length, d_model)
        self.layers = nn.ModuleList(
            [
                TemporalTransformerBlock(d_model, n_heads, feedforward_dim, dropout)
                for _ in range(n_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)
        hidden_out = max(16, d_model // 2)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, hidden_out),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_out, 1),
        )

    def forward(
        self,
        sequence_features: torch.Tensor,
        sequence_time: torch.Tensor,
        padding_mask: torch.Tensor,
        return_attention: bool = False,
    ) -> tuple[torch.Tensor, list[torch.Tensor]]:
        batch_size, sequence_length, _ = sequence_features.shape
        positions = torch.arange(sequence_length, device=sequence_features.device)
        hidden = (
            self.input_projection(sequence_features)
            + self.time_projection(sequence_time)
            + self.position_embedding(positions).unsqueeze(0).expand(batch_size, -1, -1)
        )
        causal_mask = torch.triu(
            torch.ones(sequence_length, sequence_length, device=sequence_features.device, dtype=torch.bool),
            diagonal=1,
        )
        collected_attention: list[torch.Tensor] = []
        for layer_module in self.layers:
            layer = cast(TemporalTransformerBlock, layer_module)
            hidden, attention = layer(
                hidden,
                causal_mask=causal_mask,
                padding_mask=padding_mask,
                return_attention=return_attention,
            )
            if attention is not None:
                collected_attention.append(attention)
        final_token = self.final_norm(hidden)[:, -1, :]
        logits = self.classifier(final_token).squeeze(-1)
        return logits, collected_attention


transformer_model = TemporalTransformerClassifier(
    input_dim=require_int(architecture["input_dim"], "Transformer input dimension"),
    sequence_length=SEQUENCE_LENGTH,
    d_model=require_int(architecture["d_model"], "Transformer d_model"),
    n_heads=require_int(architecture["n_heads"], "Transformer n_heads"),
    n_layers=require_int(architecture["n_layers"], "Transformer n_layers"),
    feedforward_dim=require_int(architecture["feedforward_dim"], "Transformer feedforward_dim"),
    dropout=require_float(architecture["dropout"], "Transformer dropout"),
).to(DEVICE)
transformer_model.load_state_dict(cast(dict[str, torch.Tensor], transformer_checkpoint["state_dict"]))
transformer_model.eval()

print(f"Fused transformed matrix: {fused_matrix.shape}")
print(f"Transformer sequence length: {SEQUENCE_LENGTH}")


In [ ]:
def make_transformer_batch(
    positions: np.ndarray,
    matrix: np.ndarray,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    selected_positions = np.asarray(positions, dtype=np.int64)
    feature_count = int(matrix.shape[1])
    batch_size = len(selected_positions)
    sequence_features = np.zeros((batch_size, SEQUENCE_LENGTH, feature_count), dtype=np.float32)
    sequence_time = np.zeros((batch_size, SEQUENCE_LENGTH, 1), dtype=np.float32)
    padding_mask = np.ones((batch_size, SEQUENCE_LENGTH), dtype=np.bool_)
    for batch_index, row_position in enumerate(selected_positions.tolist()):
        start = max(0, int(row_position) - SEQUENCE_LENGTH + 1)
        available = int(row_position) - start + 1
        left_padding = SEQUENCE_LENGTH - available
        sequence_features[batch_index, left_padding:] = matrix[start : int(row_position) + 1]
        sequence_time[batch_index, left_padding:, 0] = standardized_time_gap[start : int(row_position) + 1]
        padding_mask[batch_index, left_padding:] = False
    return (
        torch.as_tensor(sequence_features, dtype=torch.float32, device=DEVICE),
        torch.as_tensor(sequence_time, dtype=torch.float32, device=DEVICE),
        torch.as_tensor(padding_mask, dtype=torch.bool, device=DEVICE),
    )


def transformer_probabilities(
    positions: np.ndarray,
    matrix: np.ndarray,
    zero_feature_indices: Sequence[int] | None = None,
) -> np.ndarray:
    features_t, time_t, padding_t = make_transformer_batch(positions, matrix)
    if zero_feature_indices:
        features_t = features_t.clone()
        indices = torch.as_tensor(list(zero_feature_indices), dtype=torch.long, device=DEVICE)
        features_t.index_fill_(2, indices, 0.0)
    with torch.no_grad():
        logits, _ = transformer_model(features_t, time_t, padding_t)
        return torch.sigmoid(logits).detach().cpu().numpy().astype(float)


def transformer_global_attribution(
    positions: np.ndarray,
    noise_std: float = 0.0,
    seed: int = RANDOM_STATE,
) -> np.ndarray:
    features_t, time_t, padding_t = make_transformer_batch(positions, fused_matrix)
    features_t = features_t.detach().clone()
    if noise_std > 0.0:
        generator = torch.Generator(device=DEVICE.type)
        generator.manual_seed(seed)
        noise = torch.randn(features_t.shape, generator=generator, device=DEVICE, dtype=features_t.dtype) * noise_std
        valid = (~padding_t).unsqueeze(-1)
        features_t = features_t + noise * valid
    features_t.requires_grad_(True)
    transformer_model.zero_grad(set_to_none=True)
    logits, _ = transformer_model(features_t, time_t, padding_t)
    logits.sum().backward()
    if features_t.grad is None:
        raise RuntimeError("Transformer attribution did not produce input gradients.")
    attribution = torch.abs(features_t.grad * features_t).detach()
    valid = (~padding_t).unsqueeze(-1)
    numerator = (attribution * valid).sum(dim=(0, 1))
    denominator = valid.sum().clamp_min(1).to(attribution.dtype)
    return (numerator / denominator).cpu().numpy().astype(float)


# Regression check: reconstructed Transformer predictions must match Notebook 05.
test_global_positions = np.flatnonzero(model_data[SPLIT_COLUMN].eq("test").to_numpy(dtype=bool))
check_count = min(64, len(test_global_positions))
check_positions = test_global_positions[:check_count]
reconstructed_prob = transformer_probabilities(check_positions, fused_matrix)
saved_transformer_test = transformer_predictions_saved.loc[
    transformer_predictions_saved["split"].astype(str).eq("test")
].copy()
saved_prob_map = dict(
    zip(
        saved_transformer_test[ID_COLUMN].astype(str),
        pd.to_numeric(saved_transformer_test["predicted_probability"], errors="raise").astype(float),
    )
)
expected_prob = np.asarray(
    [saved_prob_map[str(model_data.iloc[int(position)][ID_COLUMN])] for position in check_positions],
    dtype=float,
)
transformer_max_abs_error = float(np.max(np.abs(reconstructed_prob - expected_prob))) if check_count else 0.0
if transformer_max_abs_error > 1e-5:
    raise AssertionError(
        f"Transformer reconstruction does not reproduce Notebook 05 predictions: max_abs_error={transformer_max_abs_error:.8f}"
    )
print(f"Transformer prediction reproduction max abs error: {transformer_max_abs_error:.8g}")


## 3. Reconstruct the R-GCN graph and frozen model

The R-GCN uses all candidate point events as event nodes plus the bounded, label-blind selected NF3 flow nodes. Event-to-event edges remain causal. Flow-to-event relations preserve pre-event, overlap, and post-event semantics from Notebook 06.


In [ ]:
selected_edges["event_id"] = selected_edges["event_id"].astype("string")
selected_edges["flow_id"] = selected_edges["flow_id"].astype("string")
selected_edges["relation_type"] = selected_edges["relation_type"].astype("string").str.strip().str.lower()
selected_edges["flow_start_utc"] = selected_edges["flow_start_utc"].map(
    lambda value: require_utc_timestamp(value, "flow_start_utc")
)
selected_edges["flow_end_utc"] = selected_edges["flow_end_utc"].map(
    lambda value: require_utc_timestamp(value, "flow_end_utc")
)
for column in ["signed_flow_start_lag_s", "absolute_flow_start_lag_s", "overlap_seconds", "src_port", "dst_port"]:
    if column in selected_edges.columns:
        selected_edges[column] = pd.to_numeric(selected_edges[column], errors="coerce")

flow_table = (
    selected_edges.sort_values(["flow_id", "flow_start_utc"], kind="mergesort")
    .drop_duplicates("flow_id", keep="first")
    .reset_index(drop=True)
)
flow_table["flow_duration_s"] = (
    flow_table["flow_end_utc"] - flow_table["flow_start_utc"]
).dt.total_seconds().clip(lower=0.0)
for column in ["src_port", "dst_port"]:
    if column not in flow_table.columns:
        flow_table[column] = np.nan
    flow_table[column] = pd.to_numeric(flow_table[column], errors="coerce")
if "protocol" not in flow_table.columns:
    flow_table["protocol"] = "unknown"
flow_table["protocol"] = flow_table["protocol"].astype("string").fillna("unknown")

flow_numeric = ["flow_duration_s", "src_port", "dst_port"]
flow_categorical = ["protocol"]
train_event_ids = set(model_data.loc[train_mask, ID_COLUMN].astype(str).tolist())
train_flow_ids = set(
    selected_edges.loc[
        selected_edges["event_id"].astype(str).isin(train_event_ids), "flow_id"
    ].astype(str).tolist()
)
train_flow_mask = flow_table["flow_id"].astype(str).isin(train_flow_ids)
if not train_flow_mask.any():
    raise ValueError("No selected flow nodes are adjacent to chronological training events.")

def build_rgcn_flow_preprocessor(
    numeric_columns: list[str],
    categorical_columns: list[str],
) -> ColumnTransformer:
    transformers: list[tuple[str, Pipeline, list[str]]] = []
    if numeric_columns:
        transformers.append(
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_columns,
            )
        )
    if categorical_columns:
        try:
            flow_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        except TypeError:
            flow_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
        transformers.append(
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", flow_encoder),
                    ]
                ),
                categorical_columns,
            )
        )
    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)


flow_preprocessor = build_rgcn_flow_preprocessor(flow_numeric, flow_categorical)
flow_preprocessor.fit(flow_table.loc[train_flow_mask, flow_numeric + flow_categorical])
flow_matrix = as_float32_matrix(
    flow_preprocessor.transform(flow_table[flow_numeric + flow_categorical]),
    len(flow_table),
    "R-GCN flow feature",
)
flow_feature_names = transformed_feature_names(flow_preprocessor)

if fused_matrix.shape[1] != require_int(rgcn_checkpoint["event_input_dim"], "R-GCN event_input_dim"):
    raise ValueError("Fused event matrix dimension does not match R-GCN checkpoint.")
if flow_matrix.shape[1] != require_int(rgcn_checkpoint["flow_input_dim"], "R-GCN flow_input_dim"):
    raise ValueError("Flow matrix dimension does not match R-GCN checkpoint.")


@dataclass(frozen=True)
class RelationEdges:
    source: np.ndarray
    destination: np.ndarray
    weight: np.ndarray


@dataclass(frozen=True)
class GraphBundle:
    name: str
    relations: dict[str, RelationEdges]
    event_count: int
    flow_count: int


def add_relation(
    storage: dict[str, list[tuple[np.ndarray, np.ndarray, np.ndarray]]],
    name: str,
    source: object,
    destination: object,
    weight: object,
) -> None:
    source_array = np.asarray(source, dtype=np.int64)
    destination_array = np.asarray(destination, dtype=np.int64)
    weight_array = np.asarray(weight, dtype=np.float32)
    if not (len(source_array) == len(destination_array) == len(weight_array)):
        raise ValueError(f"Relation {name!r} has inconsistent edge-array lengths.")
    if len(source_array):
        storage.setdefault(name, []).append((source_array, destination_array, weight_array))


def finalize_relations(
    storage: dict[str, list[tuple[np.ndarray, np.ndarray, np.ndarray]]]
) -> dict[str, RelationEdges]:
    output: dict[str, RelationEdges] = {}
    for relation_name, parts in storage.items():
        output[relation_name] = RelationEdges(
            source=np.concatenate([part[0] for part in parts]).astype(np.int64, copy=False),
            destination=np.concatenate([part[1] for part in parts]).astype(np.int64, copy=False),
            weight=np.concatenate([part[2] for part in parts]).astype(np.float32, copy=False),
        )
    return output


def prior_index_edges(group_values: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    positions = pd.Series(np.arange(len(group_values), dtype=np.int64), index=group_values.index)
    previous = positions.groupby(group_values.astype("string"), sort=False).shift(1)
    valid = previous.notna()
    return (
        previous.loc[valid].astype("int64").to_numpy(),
        positions.loc[valid].astype("int64").to_numpy(),
    )


def normalize_flow_relation(value: object) -> str:
    mapping = {
        "pre_event": "flow_pre_event_to_event",
        "overlaps_event_time": "flow_overlaps_event_to_event",
        "post_event": "flow_post_event_to_event",
    }
    return mapping.get(str(value).strip().lower(), "flow_other_to_event")


event_count = len(model_data)
flow_count = len(flow_table)
event_node_index = {event_id: index for index, event_id in enumerate(model_data[ID_COLUMN].astype(str).tolist())}
flow_node_index = {flow_id: event_count + index for index, flow_id in enumerate(flow_table["flow_id"].astype(str).tolist())}
lag_weight_scale = require_float(
    cast(dict[str, Any], rgcn_config["graph_policy"])["lag_weight_scale_s"],
    "R-GCN lag weight scale",
)


def build_full_rgcn_graph() -> GraphBundle:
    storage: dict[str, list[tuple[np.ndarray, np.ndarray, np.ndarray]]] = {}
    if event_count > 1:
        add_relation(
            storage,
            "site_previous_event",
            np.arange(event_count - 1, dtype=np.int64),
            np.arange(1, event_count, dtype=np.int64),
            np.ones(event_count - 1, dtype=np.float32),
        )
    aircraft_source, aircraft_destination = prior_index_edges(model_data[GROUP_COLUMN])
    add_relation(storage, "aircraft_previous_event", aircraft_source, aircraft_destination, np.ones(len(aircraft_source), dtype=np.float32))
    if EPISODE_COLUMN in model_data.columns and model_data[EPISODE_COLUMN].notna().any():
        episode_source, episode_destination = prior_index_edges(model_data[EPISODE_COLUMN])
        add_relation(storage, "episode_previous_event", episode_source, episode_destination, np.ones(len(episode_source), dtype=np.float32))

    selected = selected_edges.copy()
    selected["event_node"] = selected["event_id"].astype(str).map(event_node_index)
    selected["flow_node"] = selected["flow_id"].astype(str).map(flow_node_index)
    if selected[["event_node", "flow_node"]].isna().any().any():
        raise ValueError("Selected flow edges could not be mapped to graph nodes.")
    absolute_lag = pd.to_numeric(selected["absolute_flow_start_lag_s"], errors="raise").to_numpy(dtype=float)
    overlap = pd.to_numeric(selected["overlap_seconds"], errors="raise").to_numpy(dtype=float)
    proximity_weight = np.exp(-absolute_lag / max(lag_weight_scale, 1e-6))
    # v3.3: reconstruct with the SAME overlap scale the checkpoint was trained with. Notebook 06 v3.3 records it in
    # overlap_scale_v33.json (training-only scale); older checkpoints used the global maximum and have no record.
    import json as _json
    _scale_file = RGCN_DIR / "overlap_scale_v33.json"
    if _scale_file.exists():
        _scale_info = _json.loads(_scale_file.read_text())
        overlap_scale = float(_scale_info["overlap_scale_used_s"])
        print(f"overlap scale from checkpoint record: {overlap_scale:.3f} s (source={_scale_info.get('scale_source')})")
    else:
        overlap_scale = max(float(np.nanmax(overlap)), 1.0)
        print(f"overlap scale: global maximum {overlap_scale:.3f} s (no checkpoint record; pre-v3.3 checkpoint)")
    overlap_weight = np.clip(0.5 + 0.5 * np.log1p(overlap) / np.log1p(overlap_scale), 0.0, 1.0)
    selected["graph_weight"] = np.clip(proximity_weight * overlap_weight, 0.01, 1.0).astype(np.float32)
    selected["graph_relation"] = selected["relation_type"].map(normalize_flow_relation)
    for relation_name_obj, group in selected.groupby("graph_relation", sort=True):
        relation_name = require_text(relation_name_obj, "graph relation")
        add_relation(
            storage,
            relation_name,
            group["flow_node"].astype("int64").to_numpy(),
            group["event_node"].astype("int64").to_numpy(),
            group["graph_weight"].astype("float32").to_numpy(),
        )
    return GraphBundle(PRIMARY_RGCN_VARIANT, finalize_relations(storage), event_count, flow_count)


full_graph = build_full_rgcn_graph()
checkpoint_relations = [str(value) for value in rgcn_checkpoint.get("relation_names", [])]
if sorted(full_graph.relations) != sorted(checkpoint_relations):
    raise ValueError(
        f"Reconstructed R-GCN relations {sorted(full_graph.relations)} do not match checkpoint {sorted(checkpoint_relations)}."
    )


def graph_to_device(
    bundle: GraphBundle,
) -> dict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
    output: dict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = {}
    for relation_name, edges in bundle.relations.items():
        output[relation_name] = (
            torch.as_tensor(edges.source, dtype=torch.long, device=DEVICE),
            torch.as_tensor(edges.destination, dtype=torch.long, device=DEVICE),
            torch.as_tensor(edges.weight, dtype=torch.float32, device=DEVICE),
        )
    return output


class RelationalGraphConvolution(nn.Module):
    def __init__(self, hidden_dim: int, relation_names: Sequence[str], dropout: float) -> None:
        super().__init__()
        self.self_linear = nn.Linear(hidden_dim, hidden_dim)
        self.relation_names = tuple(str(value) for value in relation_names)
        self.relation_linears: nn.ModuleDict = nn.ModuleDict(
            {relation_name: nn.Linear(hidden_dim, hidden_dim, bias=False) for relation_name in self.relation_names}
        )
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        node_state: torch.Tensor,
        relation_tensors: Mapping[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
    ) -> torch.Tensor:
        output = self.self_linear(node_state)
        node_count_local = node_state.shape[0]
        for relation_name in self.relation_names:
            linear = cast(nn.Linear, self.relation_linears[relation_name])
            source, destination, weight = relation_tensors[relation_name]
            if source.numel() == 0:
                continue
            messages = linear(node_state.index_select(0, source)) * weight.unsqueeze(1)
            aggregate = torch.zeros_like(output)
            aggregate.index_add_(0, destination, messages)
            degree = torch.zeros(node_count_local, dtype=node_state.dtype, device=node_state.device)
            degree.index_add_(0, destination, weight.abs())
            output = output + aggregate / degree.clamp_min(1.0).unsqueeze(1)
        return self.layer_norm(self.dropout(output))


class RGCNEventClassifier(nn.Module):
    def __init__(
        self,
        event_input_dim: int,
        flow_input_dim: int,
        hidden_dim: int,
        relation_names: Sequence[str],
        layer_count: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.event_encoder = nn.Sequential(nn.Linear(event_input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.flow_encoder = nn.Sequential(nn.Linear(flow_input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.node_type_embedding = nn.Embedding(2, hidden_dim)
        self.layers: nn.ModuleList = nn.ModuleList(
            [RelationalGraphConvolution(hidden_dim, relation_names, dropout) for _ in range(layer_count)]
        )
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(
        self,
        event_features: torch.Tensor,
        flow_features: torch.Tensor,
        relation_tensors: Mapping[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
    ) -> torch.Tensor:
        event_state = self.event_encoder(event_features)
        flow_state = self.flow_encoder(flow_features)
        node_state = torch.cat([event_state, flow_state], dim=0)
        type_ids = torch.cat(
            [
                torch.zeros(event_state.shape[0], dtype=torch.long, device=node_state.device),
                torch.ones(flow_state.shape[0], dtype=torch.long, device=node_state.device),
            ]
        )
        node_state = node_state + self.node_type_embedding(type_ids)
        for layer_index in range(len(self.layers)):
            layer = cast(RelationalGraphConvolution, self.layers[layer_index])
            residual = node_state
            node_state = self.activation(layer(node_state, relation_tensors))
            node_state = self.dropout(node_state + residual)
        return self.classifier(node_state[: event_state.shape[0]]).squeeze(1)


rgcn_model = RGCNEventClassifier(
    event_input_dim=require_int(rgcn_checkpoint["event_input_dim"], "R-GCN event input dim"),
    flow_input_dim=require_int(rgcn_checkpoint["flow_input_dim"], "R-GCN flow input dim"),
    hidden_dim=require_int(rgcn_checkpoint["hidden_dim"], "R-GCN hidden dim"),
    relation_names=checkpoint_relations,
    layer_count=require_int(rgcn_checkpoint["layer_count"], "R-GCN layer count"),
    dropout=require_float(rgcn_checkpoint["dropout"], "R-GCN dropout"),
).to(DEVICE)
rgcn_model.load_state_dict(cast(dict[str, torch.Tensor], rgcn_checkpoint["model_state_dict"]))
rgcn_model.eval()

event_tensor = torch.as_tensor(fused_matrix, dtype=torch.float32, device=DEVICE)
flow_tensor = torch.as_tensor(flow_matrix, dtype=torch.float32, device=DEVICE)
full_graph_tensors = graph_to_device(full_graph)

with torch.no_grad():
    full_probabilities = torch.sigmoid(rgcn_model(event_tensor, flow_tensor, full_graph_tensors)).cpu().numpy().astype(float)

saved_rgcn_test = rgcn_predictions_saved.loc[
    rgcn_predictions_saved["split"].astype(str).eq("test")
    & rgcn_predictions_saved["variant"].astype(str).eq(PRIMARY_RGCN_VARIANT)
].copy()
saved_rgcn_prob_map = dict(
    zip(
        saved_rgcn_test[ID_COLUMN].astype(str),
        pd.to_numeric(saved_rgcn_test["probability"], errors="raise").astype(float),
    )
)
check_positions_rgcn = test_global_positions[: min(64, len(test_global_positions))]
expected_rgcn = np.asarray(
    [saved_rgcn_prob_map[str(model_data.iloc[int(position)][ID_COLUMN])] for position in check_positions_rgcn],
    dtype=float,
)
rgcn_max_abs_error = float(np.max(np.abs(full_probabilities[check_positions_rgcn] - expected_rgcn))) if len(check_positions_rgcn) else 0.0
if rgcn_max_abs_error > 1e-5:
    raise AssertionError(
        f"R-GCN reconstruction does not reproduce Notebook 06 predictions: max_abs_error={rgcn_max_abs_error:.8f}"
    )
print(f"R-GCN event matrix: {fused_matrix.shape}")
print(f"R-GCN flow matrix:  {flow_matrix.shape}")
print(f"R-GCN relations:    {sorted(full_graph.relations)}")
print(f"R-GCN prediction reproduction max abs error: {rgcn_max_abs_error:.8g}")


## 4. Global explanation synthesis

This section consolidates the existing global diagnostics into one table. Relation weight norms are treated as **diagnostic magnitude**, not causal importance. Transformer attention is treated as a temporal focus diagnostic rather than a complete causal explanation.


In [ ]:
global_rows: list[dict[str, object]] = []

transformer_attr = transformer_attribution_saved.copy()
transformer_attr["importance"] = pd.to_numeric(
    transformer_attr["mean_absolute_gradient_x_input"], errors="raise"
)
transformer_attr = transformer_attr.sort_values("importance", ascending=False, kind="mergesort")
for rank, row in enumerate(transformer_attr.head(25).itertuples(index=False), start=1):
    global_rows.append(
        {
            "model": "Temporal Transformer",
            "explanation_scope": "global",
            "evidence_type": "transformed_feature",
            "evidence_name": str(getattr(row, "transformed_feature")),
            "importance_value": require_float(getattr(row, "importance"), "Transformer importance"),
            "rank": rank,
            "interpretation": "mean absolute gradient-times-input across sampled test sequences",
        }
    )

rgcn_attr = rgcn_attribution_saved.copy()
rgcn_attr["importance"] = pd.to_numeric(
    rgcn_attr["mean_absolute_gradient_times_input"], errors="raise"
)
for node_type in ["event", "flow"]:
    subset = rgcn_attr.loc[rgcn_attr["node_type"].astype(str).eq(node_type)].sort_values(
        "importance", ascending=False, kind="mergesort"
    )
    for rank, row in enumerate(subset.head(25).itertuples(index=False), start=1):
        global_rows.append(
            {
                "model": "R-GCN",
                "explanation_scope": "global",
                "evidence_type": f"{node_type}_transformed_feature",
                "evidence_name": str(getattr(row, "feature")),
                "importance_value": require_float(getattr(row, "importance"), "R-GCN importance"),
                "rank": rank,
                "interpretation": "mean absolute gradient-times-input across sampled test event logits",
            }
        )

relation_norm_summary = (
    rgcn_relation_norms_saved.loc[
        rgcn_relation_norms_saved["variant"].astype(str).eq(PRIMARY_RGCN_VARIANT)
    ]
    .groupby("relation", as_index=False)
    .agg(importance_value=("weight_frobenius_norm", "mean"))
    .sort_values("importance_value", ascending=False, kind="mergesort")
)
for rank, row in enumerate(relation_norm_summary.itertuples(index=False), start=1):
    global_rows.append(
        {
            "model": "R-GCN",
            "explanation_scope": "global",
            "evidence_type": "relation_weight_norm",
            "evidence_name": str(getattr(row, "relation")),
            "importance_value": require_float(getattr(row, "importance_value"), "relation norm"),
            "rank": rank,
            "interpretation": "mean learned relation-transform Frobenius norm; diagnostic only",
        }
    )

if not transformer_attention_saved.empty:
    attention_frame = transformer_attention_saved.copy()
    attention_frame["mean_attention_weight"] = pd.to_numeric(attention_frame["mean_attention_weight"], errors="raise")
    attention_global = (
        attention_frame.groupby("steps_before_current", as_index=False)
        .agg(importance_value=("mean_attention_weight", "mean"))
        .sort_values("importance_value", ascending=False, kind="mergesort")
    )
    for rank, row in enumerate(attention_global.head(16).itertuples(index=False), start=1):
        step = require_int(getattr(row, "steps_before_current"), "steps before current")
        global_rows.append(
            {
                "model": "Temporal Transformer",
                "explanation_scope": "global",
                "evidence_type": "temporal_attention_position",
                "evidence_name": f"{step}_steps_before_current",
                "importance_value": require_float(getattr(row, "importance_value"), "attention weight"),
                "rank": rank,
                "interpretation": "mean final-layer final-token attention across target classes; not causal proof",
            }
        )

global_summary = pd.DataFrame(global_rows)
global_summary.to_csv(GLOBAL_SUMMARY_OUT, index=False)
display(global_summary.groupby(["model", "evidence_type"], group_keys=False).head(10))


In [ ]:
# Publication-oriented global diagnostic figures.
transformer_plot = global_summary.loc[
    global_summary["model"].eq("Temporal Transformer")
    & global_summary["evidence_type"].eq("transformed_feature")
].head(15).sort_values("importance_value", ascending=True)
if not transformer_plot.empty:
    plt.figure(figsize=(9, 6))
    plt.barh(transformer_plot["evidence_name"], transformer_plot["importance_value"])
    plt.xlabel("Mean |gradient × input|")
    plt.title("Transformer Global Feature Attribution")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "transformer_global_feature_attribution.png", dpi=220)
    plt.close()

rgcn_event_plot = global_summary.loc[
    global_summary["model"].eq("R-GCN")
    & global_summary["evidence_type"].eq("event_transformed_feature")
].head(15).sort_values("importance_value", ascending=True)
if not rgcn_event_plot.empty:
    plt.figure(figsize=(9, 6))
    plt.barh(rgcn_event_plot["evidence_name"], rgcn_event_plot["importance_value"])
    plt.xlabel("Mean |gradient × input|")
    plt.title("R-GCN Global Event-Feature Attribution")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "rgcn_global_event_feature_attribution.png", dpi=220)
    plt.close()

relation_plot = relation_norm_summary.sort_values("importance_value", ascending=True)
if not relation_plot.empty:
    plt.figure(figsize=(9, 5))
    plt.barh(relation_plot["relation"], relation_plot["importance_value"])
    plt.xlabel("Mean relation weight Frobenius norm")
    plt.title("R-GCN Relation-Weight Diagnostic")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "rgcn_relation_weight_diagnostic.png", dpi=220)
    plt.close()

if not transformer_attention_saved.empty:
    attention_plot = (
        transformer_attention_saved.assign(
            mean_attention_weight=pd.to_numeric(transformer_attention_saved["mean_attention_weight"], errors="raise")
        )
        .groupby("steps_before_current", as_index=False)
        .agg(mean_attention_weight=("mean_attention_weight", "mean"))
        .sort_values("steps_before_current")
    )
    plt.figure(figsize=(8, 5))
    plt.plot(attention_plot["steps_before_current"], attention_plot["mean_attention_weight"], marker="o")
    plt.xlabel("Steps before current event")
    plt.ylabel("Mean final-token attention")
    plt.title("Transformer Temporal Attention Diagnostic")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "transformer_temporal_attention_diagnostic.png", dpi=220)
    plt.close()


## 5. Perturbation-based fidelity

**Fidelity principle:** evidence ranked as important should materially affect the model when removed. Because the transformed feature space is centered/scaled, setting selected transformed features to zero approximates replacement with the training-reference baseline for standardized numeric features and removal of encoded categorical evidence.

The notebook compares top-ranked evidence removal with matched-size random evidence removal. For R-GCN relations, each relation is removed in turn and the held-out AUPRC and probability changes are recalculated.


In [ ]:
fidelity_rows: list[dict[str, object]] = []
rng = np.random.default_rng(RANDOM_STATE)

test_labels_all = model_data.loc[model_data[SPLIT_COLUMN].eq("test"), TARGET_COLUMN].to_numpy(dtype=np.int8)
selected_local = balanced_positions(test_labels_all, TRANSFORMER_FIDELITY_SAMPLE_MAX, RANDOM_STATE)
transformer_fidelity_positions = test_global_positions[selected_local]
transformer_fidelity_labels = model_data.iloc[transformer_fidelity_positions][TARGET_COLUMN].to_numpy(dtype=np.int8)
transformer_baseline = transformer_probabilities(transformer_fidelity_positions, fused_matrix)
transformer_baseline_auprc = safe_auprc(transformer_fidelity_labels, transformer_baseline)

transformer_importance = transformer_attr.set_index("transformed_feature")["importance"]
transformer_top_names = [name for name in transformer_importance.index.astype(str).tolist() if name in fused_feature_names][:TOP_K]
transformer_top_indices = [fused_feature_names.index(name) for name in transformer_top_names]


def add_fidelity_row(
    model_name: str,
    perturbation_type: str,
    evidence_set: str,
    repeat_index: int,
    evidence_names: Sequence[str],
    labels: np.ndarray,
    baseline_prob: np.ndarray,
    perturbed_prob: np.ndarray,
) -> None:
    baseline_auprc = safe_auprc(labels, baseline_prob)
    perturbed_auprc = safe_auprc(labels, perturbed_prob)
    positive_mask = labels == 1
    positive_drop = (
        float(np.mean(baseline_prob[positive_mask] - perturbed_prob[positive_mask]))
        if positive_mask.any()
        else float("nan")
    )
    fidelity_rows.append(
        {
            "model": model_name,
            "perturbation_type": perturbation_type,
            "evidence_set": evidence_set,
            "repeat_index": repeat_index,
            "evidence_count": len(evidence_names),
            "evidence_names": " | ".join(str(value) for value in evidence_names),
            "evaluation_rows": len(labels),
            "baseline_auprc": baseline_auprc,
            "perturbed_auprc": perturbed_auprc,
            "auprc_drop": baseline_auprc - perturbed_auprc,
            "mean_absolute_probability_change": float(np.mean(np.abs(baseline_prob - perturbed_prob))),
            "mean_probability_drop_positive_events": positive_drop,
        }
    )


transformer_top_prob = transformer_probabilities(
    transformer_fidelity_positions,
    fused_matrix,
    zero_feature_indices=transformer_top_indices,
)
add_fidelity_row(
    "Temporal Transformer",
    "feature_zeroing",
    "top_ranked",
    0,
    transformer_top_names,
    transformer_fidelity_labels,
    transformer_baseline,
    transformer_top_prob,
)

candidate_random_indices = np.asarray(
    [index for index in range(len(fused_feature_names)) if index not in transformer_top_indices],
    dtype=np.int64,
)
for repeat_index in range(1, FIDELITY_RANDOM_REPEATS + 1):
    random_indices = rng.choice(
        candidate_random_indices,
        size=min(len(transformer_top_indices), len(candidate_random_indices)),
        replace=False,
    ).astype(np.int64)
    random_names = [fused_feature_names[int(index)] for index in random_indices]
    random_prob = transformer_probabilities(
        transformer_fidelity_positions,
        fused_matrix,
        zero_feature_indices=random_indices.tolist(),
    )
    add_fidelity_row(
        "Temporal Transformer",
        "feature_zeroing",
        "random_matched",
        repeat_index,
        random_names,
        transformer_fidelity_labels,
        transformer_baseline,
        random_prob,
    )


In [ ]:
def rgcn_probabilities_with_inputs(
    event_x: torch.Tensor,
    flow_x: torch.Tensor,
    relation_tensors: Mapping[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
) -> np.ndarray:
    rgcn_model.eval()
    with torch.no_grad():
        logits = rgcn_model(event_x, flow_x, relation_tensors)
        return torch.sigmoid(logits).detach().cpu().numpy().astype(float)


def graph_with_removed_relation(
    relation_tensors: Mapping[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
    relation_to_remove: str,
) -> dict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
    output: dict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = {}
    for relation_name, (source, destination, weight) in relation_tensors.items():
        if relation_name == relation_to_remove:
            output[relation_name] = (
                torch.empty(0, dtype=torch.long, device=DEVICE),
                torch.empty(0, dtype=torch.long, device=DEVICE),
                torch.empty(0, dtype=torch.float32, device=DEVICE),
            )
        else:
            output[relation_name] = (source, destination, weight)
    return output


rgcn_test_mask_np = model_data[SPLIT_COLUMN].eq("test").to_numpy(dtype=bool)
rgcn_test_labels = model_data.loc[rgcn_test_mask_np, TARGET_COLUMN].to_numpy(dtype=np.int8)
rgcn_baseline_test_prob = full_probabilities[rgcn_test_mask_np]

# Event-feature fidelity.
rgcn_event_attr = rgcn_attr.loc[rgcn_attr["node_type"].astype(str).eq("event")].sort_values(
    "importance", ascending=False, kind="mergesort"
)
rgcn_top_event_names = [name for name in rgcn_event_attr["feature"].astype(str).tolist() if name in fused_feature_names][:TOP_K]
rgcn_top_event_indices = [fused_feature_names.index(name) for name in rgcn_top_event_names]
perturbed_event_x = event_tensor.detach().clone()
if rgcn_top_event_indices:
    perturbed_event_x[:, torch.as_tensor(rgcn_top_event_indices, dtype=torch.long, device=DEVICE)] = 0.0
rgcn_top_event_prob_all = rgcn_probabilities_with_inputs(perturbed_event_x, flow_tensor, full_graph_tensors)
add_fidelity_row(
    "R-GCN",
    "event_feature_zeroing",
    "top_ranked",
    0,
    rgcn_top_event_names,
    rgcn_test_labels,
    rgcn_baseline_test_prob,
    rgcn_top_event_prob_all[rgcn_test_mask_np],
)

remaining_event_indices = np.asarray(
    [index for index in range(len(fused_feature_names)) if index not in rgcn_top_event_indices],
    dtype=np.int64,
)
for repeat_index in range(1, FIDELITY_RANDOM_REPEATS + 1):
    random_indices = rng.choice(
        remaining_event_indices,
        size=min(len(rgcn_top_event_indices), len(remaining_event_indices)),
        replace=False,
    ).astype(np.int64)
    candidate_x = event_tensor.detach().clone()
    if len(random_indices):
        candidate_x[:, torch.as_tensor(random_indices, dtype=torch.long, device=DEVICE)] = 0.0
    candidate_prob = rgcn_probabilities_with_inputs(candidate_x, flow_tensor, full_graph_tensors)
    add_fidelity_row(
        "R-GCN",
        "event_feature_zeroing",
        "random_matched",
        repeat_index,
        [fused_feature_names[int(index)] for index in random_indices],
        rgcn_test_labels,
        rgcn_baseline_test_prob,
        candidate_prob[rgcn_test_mask_np],
    )

# Flow-feature fidelity.
rgcn_flow_attr = rgcn_attr.loc[rgcn_attr["node_type"].astype(str).eq("flow")].sort_values(
    "importance", ascending=False, kind="mergesort"
)
rgcn_top_flow_names = [name for name in rgcn_flow_attr["feature"].astype(str).tolist() if name in flow_feature_names][:TOP_K]
rgcn_top_flow_indices = [flow_feature_names.index(name) for name in rgcn_top_flow_names]
perturbed_flow_x = flow_tensor.detach().clone()
if rgcn_top_flow_indices:
    perturbed_flow_x[:, torch.as_tensor(rgcn_top_flow_indices, dtype=torch.long, device=DEVICE)] = 0.0
rgcn_top_flow_prob_all = rgcn_probabilities_with_inputs(event_tensor, perturbed_flow_x, full_graph_tensors)
add_fidelity_row(
    "R-GCN",
    "flow_feature_zeroing",
    "top_ranked",
    0,
    rgcn_top_flow_names,
    rgcn_test_labels,
    rgcn_baseline_test_prob,
    rgcn_top_flow_prob_all[rgcn_test_mask_np],
)

remaining_flow_indices = np.asarray(
    [index for index in range(len(flow_feature_names)) if index not in rgcn_top_flow_indices],
    dtype=np.int64,
)
for repeat_index in range(1, FIDELITY_RANDOM_REPEATS + 1):
    random_indices = rng.choice(
        remaining_flow_indices,
        size=min(len(rgcn_top_flow_indices), len(remaining_flow_indices)),
        replace=False,
    ).astype(np.int64)
    candidate_x = flow_tensor.detach().clone()
    if len(random_indices):
        candidate_x[:, torch.as_tensor(random_indices, dtype=torch.long, device=DEVICE)] = 0.0
    candidate_prob = rgcn_probabilities_with_inputs(event_tensor, candidate_x, full_graph_tensors)
    add_fidelity_row(
        "R-GCN",
        "flow_feature_zeroing",
        "random_matched",
        repeat_index,
        [flow_feature_names[int(index)] for index in random_indices],
        rgcn_test_labels,
        rgcn_baseline_test_prob,
        candidate_prob[rgcn_test_mask_np],
    )

# Relation-removal fidelity.
relation_removal_rows: list[dict[str, object]] = []
for relation_name in sorted(full_graph_tensors):
    removed_graph = graph_with_removed_relation(full_graph_tensors, relation_name)
    removed_prob_all = rgcn_probabilities_with_inputs(event_tensor, flow_tensor, removed_graph)
    removed_test_prob = removed_prob_all[rgcn_test_mask_np]
    add_fidelity_row(
        "R-GCN",
        "relation_removal",
        relation_name,
        0,
        [relation_name],
        rgcn_test_labels,
        rgcn_baseline_test_prob,
        removed_test_prob,
    )
    relation_removal_rows.append(
        {
            "relation": relation_name,
            "baseline_auprc": safe_auprc(rgcn_test_labels, rgcn_baseline_test_prob),
            "removed_auprc": safe_auprc(rgcn_test_labels, removed_test_prob),
            "auprc_drop": safe_auprc(rgcn_test_labels, rgcn_baseline_test_prob) - safe_auprc(rgcn_test_labels, removed_test_prob),
            "mean_absolute_probability_change": float(np.mean(np.abs(rgcn_baseline_test_prob - removed_test_prob))),
        }
    )

fidelity_results = pd.DataFrame(fidelity_rows)
fidelity_results.to_csv(FIDELITY_OUT, index=False)
display(fidelity_results)


In [ ]:
# Fidelity figure: top-ranked evidence vs matched random evidence.
fidelity_feature_plot = fidelity_results.loc[
    fidelity_results["perturbation_type"].isin(["feature_zeroing", "event_feature_zeroing", "flow_feature_zeroing"])
].copy()
if not fidelity_feature_plot.empty:
    fidelity_plot_summary = (
        fidelity_feature_plot.groupby(["model", "perturbation_type", "evidence_set"], as_index=False)
        .agg(mean_auprc_drop=("auprc_drop", "mean"))
    )
    fidelity_plot_summary["label"] = (
        fidelity_plot_summary["model"].astype(str)
        + " | "
        + fidelity_plot_summary["perturbation_type"].astype(str)
        + " | "
        + fidelity_plot_summary["evidence_set"].astype(str)
    )
    plt.figure(figsize=(10, 6))
    plt.barh(fidelity_plot_summary["label"], fidelity_plot_summary["mean_auprc_drop"])
    plt.xlabel("Mean held-out AUPRC drop after evidence removal")
    plt.title("XAI Perturbation Fidelity")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "xai_fidelity_comparison.png", dpi=220)
    plt.close()


## 6. Explanation stability under small input perturbations

Stability is evaluated by adding small Gaussian perturbations in the transformed feature space and recomputing gradient-times-input rankings. The notebook reports top-*k* Jaccard overlap and Spearman rank correlation against the unperturbed explanation.


In [ ]:
stability_rows: list[dict[str, object]] = []

# Transformer global stability on a balanced test sample.
stability_local = balanced_positions(test_labels_all, TRANSFORMER_STABILITY_SAMPLE_MAX, RANDOM_STATE + 10)
transformer_stability_positions = test_global_positions[stability_local]
transformer_reference_attr = transformer_global_attribution(transformer_stability_positions, noise_std=0.0)
for repeat_index in range(1, STABILITY_REPEATS + 1):
    candidate_attr = transformer_global_attribution(
        transformer_stability_positions,
        noise_std=NOISE_STD,
        seed=RANDOM_STATE + 100 + repeat_index,
    )
    stability_rows.append(
        {
            "model": "Temporal Transformer",
            "evidence_type": "transformed_feature",
            "repeat_index": repeat_index,
            "noise_std": NOISE_STD,
            "top_k": TOP_K,
            "top_k_jaccard": top_k_jaccard(transformer_reference_attr, candidate_attr, TOP_K),
            "spearman_rank_correlation": spearman_rank_correlation(transformer_reference_attr, candidate_attr),
            "evaluation_rows": len(transformer_stability_positions),
        }
    )


def rgcn_global_attribution(
    event_x_base: torch.Tensor,
    flow_x_base: torch.Tensor,
    target_indices: np.ndarray,
    noise_std: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    event_x = event_x_base.detach().clone()
    flow_x = flow_x_base.detach().clone()
    if noise_std > 0.0:
        generator = torch.Generator(device=DEVICE.type)
        generator.manual_seed(seed)
        event_x = event_x + torch.randn(event_x.shape, generator=generator, device=DEVICE, dtype=event_x.dtype) * noise_std
        flow_x = flow_x + torch.randn(flow_x.shape, generator=generator, device=DEVICE, dtype=flow_x.dtype) * noise_std
    event_x.requires_grad_(True)
    flow_x.requires_grad_(True)
    rgcn_model.zero_grad(set_to_none=True)
    logits = rgcn_model(event_x, flow_x, full_graph_tensors)
    target_tensor = torch.as_tensor(target_indices, dtype=torch.long, device=DEVICE)
    logits.index_select(0, target_tensor).sum().backward()
    if event_x.grad is None or flow_x.grad is None:
        raise RuntimeError("R-GCN stability attribution did not produce gradients.")
    event_attr = torch.abs(event_x.grad * event_x).mean(dim=0).detach().cpu().numpy().astype(float)
    flow_attr = torch.abs(flow_x.grad * flow_x).mean(dim=0).detach().cpu().numpy().astype(float)
    return event_attr, flow_attr


rgcn_stability_local = balanced_positions(test_labels_all, RGCN_ATTRIBUTION_SAMPLE_MAX, RANDOM_STATE + 20)
rgcn_stability_positions = test_global_positions[rgcn_stability_local]
rgcn_reference_event_attr, rgcn_reference_flow_attr = rgcn_global_attribution(
    event_tensor,
    flow_tensor,
    rgcn_stability_positions,
    noise_std=0.0,
    seed=RANDOM_STATE,
)
for repeat_index in range(1, STABILITY_REPEATS + 1):
    event_candidate, flow_candidate = rgcn_global_attribution(
        event_tensor,
        flow_tensor,
        rgcn_stability_positions,
        noise_std=NOISE_STD,
        seed=RANDOM_STATE + 200 + repeat_index,
    )
    stability_rows.append(
        {
            "model": "R-GCN",
            "evidence_type": "event_transformed_feature",
            "repeat_index": repeat_index,
            "noise_std": NOISE_STD,
            "top_k": TOP_K,
            "top_k_jaccard": top_k_jaccard(rgcn_reference_event_attr, event_candidate, TOP_K),
            "spearman_rank_correlation": spearman_rank_correlation(rgcn_reference_event_attr, event_candidate),
            "evaluation_rows": len(rgcn_stability_positions),
        }
    )
    stability_rows.append(
        {
            "model": "R-GCN",
            "evidence_type": "flow_transformed_feature",
            "repeat_index": repeat_index,
            "noise_std": NOISE_STD,
            "top_k": TOP_K,
            "top_k_jaccard": top_k_jaccard(rgcn_reference_flow_attr, flow_candidate, TOP_K),
            "spearman_rank_correlation": spearman_rank_correlation(rgcn_reference_flow_attr, flow_candidate),
            "evaluation_rows": len(rgcn_stability_positions),
        }
    )

stability_results = pd.DataFrame(stability_rows)
stability_results.to_csv(STABILITY_OUT, index=False)
display(stability_results)


In [ ]:
if not stability_results.empty:
    stability_plot = (
        stability_results.groupby(["model", "evidence_type"], as_index=False)
        .agg(mean_top_k_jaccard=("top_k_jaccard", "mean"))
    )
    stability_plot["label"] = stability_plot["model"].astype(str) + " | " + stability_plot["evidence_type"].astype(str)
    plt.figure(figsize=(9, 5))
    plt.barh(stability_plot["label"], stability_plot["mean_top_k_jaccard"])
    plt.xlim(0.0, 1.0)
    plt.xlabel(f"Mean top-{TOP_K} Jaccard overlap")
    plt.title("XAI Stability Under Small Input Perturbations")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "xai_stability_comparison.png", dpi=220)
    plt.close()


## 7. Relation-diagnostic consistency

This test asks whether learned R-GCN relation diagnostics agree with a stronger behavioral test: **what happens to held-out performance when that relation is removed?** Agreement is summarized with rank correlation but is not interpreted as causal proof.


In [ ]:
relation_removal = pd.DataFrame(relation_removal_rows)
relation_norm = relation_norm_summary.rename(columns={"importance_value": "mean_weight_frobenius_norm"})
relation_consistency = relation_norm.merge(relation_removal, on="relation", how="inner", validate="one_to_one")
if relation_consistency.empty:
    raise ValueError("No R-GCN relations were available for consistency analysis.")
relation_consistency["norm_rank"] = relation_consistency["mean_weight_frobenius_norm"].rank(method="average", ascending=False)
relation_consistency["auprc_drop_rank"] = relation_consistency["auprc_drop"].rank(method="average", ascending=False)
relation_consistency["probability_change_rank"] = relation_consistency["mean_absolute_probability_change"].rank(method="average", ascending=False)
relation_spearman_auprc = spearman_rank_correlation(
    relation_consistency["mean_weight_frobenius_norm"].to_numpy(dtype=float),
    relation_consistency["auprc_drop"].to_numpy(dtype=float),
)
relation_spearman_probability = spearman_rank_correlation(
    relation_consistency["mean_weight_frobenius_norm"].to_numpy(dtype=float),
    relation_consistency["mean_absolute_probability_change"].to_numpy(dtype=float),
)
relation_consistency["global_spearman_norm_vs_auprc_drop"] = relation_spearman_auprc
relation_consistency["global_spearman_norm_vs_probability_change"] = relation_spearman_probability
relation_consistency.to_csv(RELATION_CONSISTENCY_OUT, index=False)
display(relation_consistency.sort_values("auprc_drop", ascending=False))
print(f"Spearman(norm, AUPRC drop):             {relation_spearman_auprc:.4f}")
print(f"Spearman(norm, probability change):     {relation_spearman_probability:.4f}")


In [ ]:
plt.figure(figsize=(8, 5))
plot_rel = relation_consistency.sort_values("auprc_drop", ascending=True)
plt.barh(plot_rel["relation"], plot_rel["auprc_drop"])
plt.xlabel("Held-out AUPRC drop when relation is removed")
plt.title("R-GCN Relation-Removal Fidelity")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "rgcn_relation_removal_fidelity.png", dpi=220)
plt.close()


## 8. Local causal-subgraph explanations and analyst case selection

Representative cases are selected from the held-out test partition without changing any model threshold. Local R-GCN explanations use only the causal incoming subgraph available to the target event. Each case preserves source identifiers for analyst traceability.


In [ ]:
def build_incoming_adjacency(
    bundle: GraphBundle,
) -> dict[int, list[tuple[str, int, float]]]:
    incoming: dict[int, list[tuple[str, int, float]]] = {}
    for relation_name, edges in bundle.relations.items():
        for source, destination, weight in zip(
            edges.source.tolist(), edges.destination.tolist(), edges.weight.tolist()
        ):
            destination_int = int(destination)
            incoming.setdefault(destination_int, []).append((relation_name, int(source), float(weight)))
    return incoming


incoming_adjacency = build_incoming_adjacency(full_graph)


def make_local_subgraph(
    target_event_index: int,
    hops: int = LOCAL_HOPS,
) -> tuple[
    torch.Tensor,
    torch.Tensor,
    dict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
    int,
    list[int],
    list[int],
]:
    selected_nodes: set[int] = {int(target_event_index)}
    frontier: set[int] = {int(target_event_index)}
    for _ in range(max(1, hops)):
        next_frontier: set[int] = set()
        for destination in frontier:
            for _, source, _ in incoming_adjacency.get(destination, []):
                if source not in selected_nodes:
                    selected_nodes.add(source)
                    next_frontier.add(source)
        frontier = next_frontier
        if not frontier:
            break

    local_event_global = sorted(index for index in selected_nodes if index < event_count)
    local_flow_global = sorted(index for index in selected_nodes if index >= event_count)
    ordered_global = local_event_global + local_flow_global
    remap = {global_index: local_index for local_index, global_index in enumerate(ordered_global)}

    relation_tensors: dict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = {}
    for relation_name in sorted(full_graph.relations):
        edges = full_graph.relations[relation_name]
        parts: list[tuple[int, int, float]] = []
        for source, destination, weight in zip(edges.source.tolist(), edges.destination.tolist(), edges.weight.tolist()):
            source_int = int(source)
            destination_int = int(destination)
            if source_int in remap and destination_int in remap:
                parts.append((remap[source_int], remap[destination_int], float(weight)))
        if parts:
            relation_tensors[relation_name] = (
                torch.tensor([part[0] for part in parts], dtype=torch.long, device=DEVICE),
                torch.tensor([part[1] for part in parts], dtype=torch.long, device=DEVICE),
                torch.tensor([part[2] for part in parts], dtype=torch.float32, device=DEVICE),
            )
        else:
            relation_tensors[relation_name] = (
                torch.empty(0, dtype=torch.long, device=DEVICE),
                torch.empty(0, dtype=torch.long, device=DEVICE),
                torch.empty(0, dtype=torch.float32, device=DEVICE),
            )

    local_event_x = torch.as_tensor(
        fused_matrix[np.asarray(local_event_global, dtype=np.int64)],
        dtype=torch.float32,
        device=DEVICE,
    )
    local_flow_positions = np.asarray([index - event_count for index in local_flow_global], dtype=np.int64)
    local_flow_x = torch.as_tensor(
        flow_matrix[local_flow_positions],
        dtype=torch.float32,
        device=DEVICE,
    )
    target_local_event_index = local_event_global.index(int(target_event_index))
    return (
        local_event_x,
        local_flow_x,
        relation_tensors,
        target_local_event_index,
        local_event_global,
        local_flow_global,
    )


def explain_transformer_event(event_position: int) -> tuple[float, list[str], float]:
    features_t, time_t, padding_t = make_transformer_batch(np.asarray([event_position], dtype=np.int64), fused_matrix)
    features_t = features_t.detach().clone().requires_grad_(True)
    synchronize_device()
    started = time.perf_counter()
    transformer_model.zero_grad(set_to_none=True)
    logits, _ = transformer_model(features_t, time_t, padding_t, return_attention=True)
    probability = torch.sigmoid(logits[0])
    logits[0].backward()
    synchronize_device()
    elapsed_ms = (time.perf_counter() - started) * 1000.0
    if features_t.grad is None:
        raise RuntimeError("Local Transformer explanation produced no gradients.")
    attribution = torch.abs(features_t.grad * features_t).detach()[0]
    valid = (~padding_t[0]).unsqueeze(-1)
    feature_scores = (attribution * valid).sum(dim=0).cpu().numpy().astype(float)
    top_indices = np.argsort(-feature_scores)[: min(5, len(feature_scores))]
    top_features = [fused_feature_names[int(index)] for index in top_indices]
    return float(probability.detach().cpu().item()), top_features, elapsed_ms


def explain_rgcn_event(
    event_position: int,
) -> tuple[float, list[str], list[str], list[str], float, int, int]:
    synchronize_device()
    started = time.perf_counter()
    local_event_x, local_flow_x, local_relations, target_local, local_event_global, local_flow_global = make_local_subgraph(event_position)
    local_event_x = local_event_x.detach().clone().requires_grad_(True)
    local_flow_x = local_flow_x.detach().clone().requires_grad_(True)
    rgcn_model.zero_grad(set_to_none=True)
    logits = rgcn_model(local_event_x, local_flow_x, local_relations)
    target_logit = logits[target_local]
    probability = torch.sigmoid(target_logit)
    target_logit.backward()
    synchronize_device()
    elapsed_ms = (time.perf_counter() - started) * 1000.0

    if local_event_x.grad is None or local_flow_x.grad is None:
        raise RuntimeError("Local R-GCN explanation produced no gradients.")
    event_scores = torch.abs(local_event_x.grad * local_event_x).mean(dim=0).detach().cpu().numpy().astype(float)
    flow_scores = torch.abs(local_flow_x.grad * local_flow_x).mean(dim=0).detach().cpu().numpy().astype(float)
    event_top_idx = np.argsort(-event_scores)[: min(5, len(event_scores))]
    flow_top_idx = np.argsort(-flow_scores)[: min(5, len(flow_scores))]
    event_top_features = [fused_feature_names[int(index)] for index in event_top_idx]
    flow_top_features = [flow_feature_names[int(index)] for index in flow_top_idx]

    relation_changes: list[tuple[str, float]] = []
    baseline_probability = float(probability.detach().cpu().item())
    with torch.no_grad():
        for relation_name in sorted(local_relations):
            removed = graph_with_removed_relation(local_relations, relation_name)
            candidate_probability = torch.sigmoid(
                rgcn_model(local_event_x.detach(), local_flow_x.detach(), removed)[target_local]
            )
            relation_changes.append(
                (relation_name, abs(baseline_probability - float(candidate_probability.detach().cpu().item())))
            )
    relation_changes.sort(key=lambda item: item[1], reverse=True)
    top_relations = [name for name, _ in relation_changes[: min(5, len(relation_changes))]]
    return (
        baseline_probability,
        event_top_features,
        flow_top_features,
        top_relations,
        elapsed_ms,
        len(local_event_global),
        len(local_flow_global),
    )


In [ ]:
# Build aligned test prediction bank for case selection.
transformer_test = transformer_predictions_saved.loc[
    transformer_predictions_saved["split"].astype(str).eq("test")
    & transformer_predictions_saved["feature_set"].astype(str).eq(PRIMARY_TRANSFORMER_FEATURE_SET)
, [ID_COLUMN, "model_target", "predicted_probability", "selected_threshold", "predicted_label"]].copy()
transformer_test = transformer_test.rename(
    columns={
        "predicted_probability": "transformer_probability",
        "selected_threshold": "transformer_threshold",
        "predicted_label": "transformer_predicted_label",
    }
)
rgcn_test = rgcn_predictions_saved.loc[
    rgcn_predictions_saved["split"].astype(str).eq("test")
    & rgcn_predictions_saved["variant"].astype(str).eq(PRIMARY_RGCN_VARIANT)
, [ID_COLUMN, "probability", "selected_threshold", "predicted_label"]].copy()
rgcn_test = rgcn_test.rename(
    columns={
        "probability": "rgcn_probability",
        "selected_threshold": "rgcn_threshold",
        "predicted_label": "rgcn_predicted_label",
    }
)
case_bank = transformer_test.merge(rgcn_test, on=ID_COLUMN, how="inner", validate="one_to_one")
case_bank["model_target"] = pd.to_numeric(case_bank["model_target"], errors="raise").astype("int8")
case_bank["transformer_predicted_label"] = pd.to_numeric(case_bank["transformer_predicted_label"], errors="raise").astype("int8")
case_bank["rgcn_predicted_label"] = pd.to_numeric(case_bank["rgcn_predicted_label"], errors="raise").astype("int8")
case_bank["probability_difference"] = (
    pd.to_numeric(case_bank["rgcn_probability"], errors="raise")
    - pd.to_numeric(case_bank["transformer_probability"], errors="raise")
).abs()

def first_distinct(frame: pd.DataFrame, used: set[str], sort_column: str, ascending: bool) -> str | None:
    ordered = frame.sort_values(sort_column, ascending=ascending, kind="mergesort")
    for value in ordered[ID_COLUMN].astype(str).tolist():
        if value not in used:
            return value
    return None

case_specs: list[tuple[str, pd.DataFrame, str, bool]] = [
    (
        "true_positive_high_confidence",
        case_bank.loc[(case_bank["model_target"] == 1) & (case_bank["rgcn_predicted_label"] == 1)],
        "rgcn_probability",
        False,
    ),
    (
        "false_positive_high_confidence",
        case_bank.loc[(case_bank["model_target"] == 0) & (case_bank["rgcn_predicted_label"] == 1)],
        "rgcn_probability",
        False,
    ),
    (
        "false_negative_low_confidence",
        case_bank.loc[(case_bank["model_target"] == 1) & (case_bank["rgcn_predicted_label"] == 0)],
        "rgcn_probability",
        True,
    ),
    (
        "model_disagreement",
        case_bank.loc[case_bank["transformer_predicted_label"] != case_bank["rgcn_predicted_label"]],
        "probability_difference",
        False,
    ),
    (
        "true_negative_high_confidence",
        case_bank.loc[
            (case_bank["model_target"] == 0)
            & (case_bank["rgcn_predicted_label"] == 0)
            & (case_bank["transformer_predicted_label"] == 0)
        ],
        "rgcn_probability",
        True,
    ),
]

selected_cases: list[tuple[str, str]] = []
used_event_ids: set[str] = set()
for case_type, candidate_frame, sort_column, ascending in case_specs:
    event_id = first_distinct(candidate_frame, used_event_ids, sort_column, ascending)
    if event_id is not None:
        selected_cases.append((case_type, event_id))
        used_event_ids.add(event_id)
    if len(selected_cases) >= CASE_STUDY_COUNT:
        break

# Fill any missing slots with largest model disagreements.
if len(selected_cases) < CASE_STUDY_COUNT:
    for event_id in case_bank.sort_values("probability_difference", ascending=False)[ID_COLUMN].astype(str).tolist():
        if event_id not in used_event_ids:
            selected_cases.append(("supplemental_large_probability_difference", event_id))
            used_event_ids.add(event_id)
        if len(selected_cases) >= CASE_STUDY_COUNT:
            break

print("Selected analyst cases:")
for case_type, event_id in selected_cases:
    print(f"  {case_type}: {event_id}")


In [ ]:
case_rows: list[dict[str, object]] = []
latency_sample_rows: list[dict[str, object]] = []
model_position_map = {event_id: index for index, event_id in enumerate(model_data[ID_COLUMN].astype(str).tolist())}

for case_type, event_id in selected_cases:
    event_position = model_position_map[event_id]
    event_row = model_data.iloc[event_position]
    prediction_row = case_bank.loc[case_bank[ID_COLUMN].astype(str).eq(event_id)].iloc[0]

    transformer_probability_local, transformer_top_features, transformer_latency_ms = explain_transformer_event(event_position)
    (
        rgcn_probability_local,
        rgcn_event_features,
        rgcn_flow_features,
        rgcn_top_relations,
        rgcn_latency_ms,
        local_event_nodes,
        local_flow_nodes,
    ) = explain_rgcn_event(event_position)

    event_edges = selected_edges.loc[selected_edges["event_id"].astype(str).eq(event_id)].copy()
    event_edges = event_edges.sort_values(["absolute_flow_start_lag_s", "overlap_seconds"], ascending=[True, False], kind="mergesort")
    nearest_flow_ids = event_edges["flow_id"].astype(str).head(3).tolist()
    nearest_flow_lags = pd.to_numeric(event_edges["signed_flow_start_lag_s"], errors="coerce").head(3).tolist()
    relation_counts = event_edges["relation_type"].astype(str).value_counts().to_dict()

    previous_site_event_id = str(model_data.iloc[event_position - 1][ID_COLUMN]) if event_position > 0 else ""
    same_aircraft_prior = model_data.iloc[:event_position].loc[
        model_data.iloc[:event_position][GROUP_COLUMN].astype(str).eq(str(event_row[GROUP_COLUMN]))
    ]
    previous_same_aircraft_id = str(same_aircraft_prior.iloc[-1][ID_COLUMN]) if not same_aircraft_prior.empty else ""

    case_rows.append(
        {
            "case_type": case_type,
            "event_id": event_id,
            "event_time_utc": str(event_row[TIME_COLUMN]),
            "aircraft_key": str(event_row[GROUP_COLUMN]),
            "model_target": require_int(prediction_row["model_target"], "case target"),
            "transformer_probability": require_float(prediction_row["transformer_probability"], "Transformer probability"),
            "transformer_threshold": require_float(prediction_row["transformer_threshold"], "Transformer threshold"),
            "transformer_predicted_label": require_int(prediction_row["transformer_predicted_label"], "Transformer predicted label"),
            "rgcn_probability": require_float(prediction_row["rgcn_probability"], "R-GCN probability"),
            "rgcn_threshold": require_float(prediction_row["rgcn_threshold"], "R-GCN threshold"),
            "rgcn_predicted_label": require_int(prediction_row["rgcn_predicted_label"], "R-GCN predicted label"),
            "altitude_ft": event_row.get("altitude_ft", np.nan),
            "ground_speed_kt": event_row.get("ground_speed_kt", np.nan),
            "distance_miles": event_row.get("distance_miles", np.nan),
            "event_lag_seconds": event_row.get("event_lag_seconds", np.nan),
            "stale_event_flag": event_row.get("stale_event_flag", np.nan),
            "source_type": event_row.get("source_type", ""),
            "prior_event_count": event_row.get("prior_event_count", np.nan),
            "recurrence_24h_flag": event_row.get("recurrence_24h_flag", np.nan),
            "episode_id": event_row.get("episode_id", ""),
            "episode_size": event_row.get("episode_size", np.nan),
            "previous_site_event_id": previous_site_event_id,
            "previous_same_aircraft_event_id": previous_same_aircraft_id,
            "selected_flow_count": len(event_edges),
            "flow_relation_counts": json.dumps(relation_counts, sort_keys=True),
            "nearest_flow_ids": " | ".join(nearest_flow_ids),
            "nearest_flow_signed_lag_s": " | ".join(str(value) for value in nearest_flow_lags),
            "transformer_top_local_features": " | ".join(transformer_top_features),
            "rgcn_top_local_event_features": " | ".join(rgcn_event_features),
            "rgcn_top_local_flow_features": " | ".join(rgcn_flow_features),
            "rgcn_top_local_relations": " | ".join(rgcn_top_relations),
            "transformer_explanation_latency_ms": transformer_latency_ms,
            "rgcn_explanation_latency_ms": rgcn_latency_ms,
            "rgcn_local_event_nodes": local_event_nodes,
            "rgcn_local_flow_nodes": local_flow_nodes,
            "interpretation_boundary": (
                "Controlled cross-domain replay. Explanations indicate model influence, not causal linkage, "
                "site coincidence, or proof that the aircraft generated the network activity."
            ),
        }
    )

case_studies = pd.DataFrame(case_rows)
case_studies.to_csv(CASE_STUDIES_OUT, index=False)
display(case_studies)


In [ ]:
# Write one concise analyst evidence card per selected case.
for row in case_studies.itertuples(index=False):
    event_id = str(getattr(row, "event_id"))
    case_type = str(getattr(row, "case_type"))
    card_text = f"""# Analyst Evidence Card — {event_id}

**Case type:** {case_type}  
**Event time:** {getattr(row, 'event_time_utc')}  
**Aircraft key:** {getattr(row, 'aircraft_key')}  
**Altitude:** {getattr(row, 'altitude_ft')} ft  
**Ground speed:** {getattr(row, 'ground_speed_kt')} kt  
**Distance from PANC:** {getattr(row, 'distance_miles')} miles

## Model outputs

- Transformer probability: {require_float(getattr(row, 'transformer_probability'), 'Transformer card probability'):.6f}
- Transformer threshold: {require_float(getattr(row, 'transformer_threshold'), 'Transformer card threshold'):.6f}
- R-GCN probability: {require_float(getattr(row, 'rgcn_probability'), 'R-GCN card probability'):.6f}
- R-GCN threshold: {require_float(getattr(row, 'rgcn_threshold'), 'R-GCN card threshold'):.6f}
- Evaluation target: {require_int(getattr(row, 'model_target'), 'card target')}

## Explanation evidence

**Transformer top local features:** {getattr(row, 'transformer_top_local_features')}  
**R-GCN top event features:** {getattr(row, 'rgcn_top_local_event_features')}  
**R-GCN top flow features:** {getattr(row, 'rgcn_top_local_flow_features')}  
**R-GCN top local relations:** {getattr(row, 'rgcn_top_local_relations')}

## Traceable context

- Previous site event: {getattr(row, 'previous_site_event_id')}
- Previous same-aircraft event: {getattr(row, 'previous_same_aircraft_event_id')}
- Selected flow count: {getattr(row, 'selected_flow_count')}
- Nearest flow IDs: {getattr(row, 'nearest_flow_ids')}
- Nearest signed flow lags (s): {getattr(row, 'nearest_flow_signed_lag_s')}

## Interpretation boundary

{getattr(row, 'interpretation_boundary')}
"""
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{case_type}_{event_id}")
    (ANALYST_CARDS_DIR / f"{safe_name}.md").write_text(card_text, encoding="utf-8")

print(f"Analyst cards written: {len(list(ANALYST_CARDS_DIR.glob('*.md')))}")


## 9. Explanation latency

Latency is measured separately from prediction latency because an analyst-facing system must account for the additional time required to generate an explanation. Transformer latency measures a local forward/backward attribution pass. R-GCN latency includes local causal-subgraph construction and a local forward/backward attribution pass.


In [ ]:
# Use representative cases first, then supplement with balanced test events.
latency_event_ids = [event_id for _, event_id in selected_cases]
if len(latency_event_ids) < LATENCY_SAMPLE_MAX:
    supplement_local = balanced_positions(test_labels_all, LATENCY_SAMPLE_MAX, RANDOM_STATE + 30)
    for global_position in test_global_positions[supplement_local].tolist():
        event_id = str(model_data.iloc[int(global_position)][ID_COLUMN])
        if event_id not in latency_event_ids:
            latency_event_ids.append(event_id)
        if len(latency_event_ids) >= LATENCY_SAMPLE_MAX:
            break

for event_id in latency_event_ids[:LATENCY_SAMPLE_MAX]:
    event_position = model_position_map[event_id]
    _, _, transformer_latency_ms = explain_transformer_event(event_position)
    _, _, _, _, rgcn_latency_ms, local_event_nodes, local_flow_nodes = explain_rgcn_event(event_position)
    latency_sample_rows.extend(
        [
            {
                "model": "Temporal Transformer",
                "event_id": event_id,
                "measurement_type": "local_forward_backward_attribution",
                "latency_ms": transformer_latency_ms,
                "local_event_nodes": np.nan,
                "local_flow_nodes": np.nan,
            },
            {
                "model": "R-GCN",
                "event_id": event_id,
                "measurement_type": "local_subgraph_build_plus_forward_backward_attribution",
                "latency_ms": rgcn_latency_ms,
                "local_event_nodes": local_event_nodes,
                "local_flow_nodes": local_flow_nodes,
            },
        ]
    )

latency_samples = pd.DataFrame(latency_sample_rows)
latency_samples.to_csv(LATENCY_SAMPLES_OUT, index=False)
latency_summary = (
    latency_samples.groupby(["model", "measurement_type"], as_index=False)
    .agg(
        samples=("latency_ms", "size"),
        latency_mean_ms=("latency_ms", "mean"),
        latency_p50_ms=("latency_ms", lambda values: float(np.percentile(np.asarray(values, dtype=float), 50))),
        latency_p95_ms=("latency_ms", lambda values: float(np.percentile(np.asarray(values, dtype=float), 95))),
    )
)
latency_summary.to_csv(LATENCY_OUT, index=False)
display(latency_summary)


In [ ]:
if not latency_summary.empty:
    latency_plot = latency_summary.sort_values("latency_p95_ms", ascending=True)
    plt.figure(figsize=(8, 4.5))
    plt.barh(latency_plot["model"], latency_plot["latency_p95_ms"])
    plt.xlabel("p95 explanation latency (ms)")
    plt.title("Analyst Explanation Latency")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "xai_explanation_latency.png", dpi=220)
    plt.close()


## 10. Provenance audit and model-explanation matrix

The provenance audit verifies that analyst evidence can be traced back to frozen point-event and flow identifiers. The model matrix summarizes what each model can and cannot explain in the present implementation.


In [ ]:
provenance_rows: list[dict[str, object]] = []

def add_audit(
    check_name: str,
    expected: object,
    actual: object,
    passed: object,
    note: str,
) -> None:
    if not isinstance(passed, (bool, np.bool_)):
        raise TypeError(
            f"{check_name}: passed must be bool-like, got {type(passed).__name__}."
        )

    normalized_passed = bool(passed)

    provenance_rows.append(
        {
            "check": check_name,
            "expected": expected,
            "actual": actual,
            "passed": normalized_passed,
            "note": note,
        }
    )

feature_event_ids = set(model_data[ID_COLUMN].astype(str).tolist())
node_event_ids = set(node_manifest.loc[node_manifest["node_type"].astype(str).eq("event"), "node_id"].astype(str).tolist())
node_flow_ids = set(node_manifest.loc[node_manifest["node_type"].astype(str).eq("flow"), "node_id"].astype(str).tolist())
selected_event_ids = set(selected_edges["event_id"].astype(str).tolist())
selected_flow_ids = set(selected_edges["flow_id"].astype(str).tolist())
case_event_ids = set(case_studies["event_id"].astype(str).tolist())

add_audit("all_point_events_have_event_nodes", len(feature_event_ids), len(feature_event_ids & node_event_ids), feature_event_ids == node_event_ids, "Point-event IDs must map one-to-one to R-GCN event nodes.")
add_audit("selected_edge_event_ids_trace_to_point_events", len(selected_event_ids), len(selected_event_ids & feature_event_ids), selected_event_ids.issubset(feature_event_ids), "Every selected edge must point to a frozen candidate event.")
add_audit("selected_flow_ids_trace_to_flow_nodes", len(selected_flow_ids), len(selected_flow_ids & node_flow_ids), selected_flow_ids.issubset(node_flow_ids), "Every selected flow must map to an R-GCN flow node.")
add_audit("analyst_case_event_ids_trace_to_point_events", len(case_event_ids), len(case_event_ids & feature_event_ids), case_event_ids.issubset(feature_event_ids), "Every analyst case must trace to a source point event.")
add_audit("transformer_feature_names_match_checkpoint", len(checkpoint_feature_names), len(fused_feature_names), checkpoint_feature_names == fused_feature_names, "Reconstructed feature identities must match the frozen checkpoint.")
add_audit("transformer_predictions_reproduced", "max abs error <= 1e-5", transformer_max_abs_error, transformer_max_abs_error <= 1e-5, "Perturbation tests are valid only if the frozen prediction is reproduced.")
add_audit("rgcn_predictions_reproduced", "max abs error <= 1e-5", rgcn_max_abs_error, rgcn_max_abs_error <= 1e-5, "Perturbation tests are valid only if the frozen prediction is reproduced.")
add_audit("controlled_alignment_disclaimer_present", True, case_studies["interpretation_boundary"].astype(str).str.contains("Controlled cross-domain replay", regex=False).all(), case_studies["interpretation_boundary"].astype(str).str.contains("Controlled cross-domain replay", regex=False).all(), "Every analyst case must preserve the controlled-replay interpretation boundary.")

provenance_audit = pd.DataFrame(provenance_rows)
provenance_audit.to_csv(PROVENANCE_OUT, index=False)
provenance_coverage = float(provenance_audit["passed"].mean()) if len(provenance_audit) else float("nan")
display(provenance_audit)
print(f"Provenance audit pass fraction: {provenance_coverage:.3f}")


In [ ]:
model_matrix = pd.DataFrame(
    [
        {
            "model": "Tabular fused baseline",
            "global_feature_explanation": "Yes — coefficients/tree/permutation importance from Notebook 04",
            "local_feature_explanation": "Not added in Notebook 09",
            "temporal_context_explanation": "Limited to engineered temporal features",
            "relation_explanation": "No",
            "individual_flow_evidence": "No — cyber evidence is aggregated into event-window features",
            "fidelity_tested_in_notebook09": "No",
            "stability_tested_in_notebook09": "No",
            "provenance_audited": "Yes at event-package level",
            "explanation_latency_measured": "No",
        },
        {
            "model": "Temporal Transformer",
            "global_feature_explanation": "Yes — gradient-times-input",
            "local_feature_explanation": "Yes — event-specific gradient-times-input",
            "temporal_context_explanation": "Yes — causal sequence plus attention diagnostic",
            "relation_explanation": "No explicit typed graph relations",
            "individual_flow_evidence": "No — cyber inputs are event-window features",
            "fidelity_tested_in_notebook09": "Yes — top-vs-random feature removal",
            "stability_tested_in_notebook09": "Yes — small transformed-input perturbations",
            "provenance_audited": "Yes",
            "explanation_latency_measured": "Yes",
        },
        {
            "model": "R-GCN",
            "global_feature_explanation": "Yes — event and flow gradient-times-input",
            "local_feature_explanation": "Yes — local causal-subgraph attribution",
            "temporal_context_explanation": "Yes — directed historical event relations and lag-aware flow relations",
            "relation_explanation": "Yes — relation norms plus relation-removal fidelity",
            "individual_flow_evidence": "Yes — selected flow IDs and lags remain traceable",
            "fidelity_tested_in_notebook09": "Yes — feature and relation removal",
            "stability_tested_in_notebook09": "Yes — small transformed-input perturbations",
            "provenance_audited": "Yes",
            "explanation_latency_measured": "Yes",
        },
    ]
)
model_matrix.to_csv(MODEL_MATRIX_OUT, index=False)
display(model_matrix)


## 11. Consolidated XAI reference, narrative, figures, and checksums

The final artifacts summarize what the XAI layer demonstrated without claiming that attribution equals causality or that analyst trust was empirically improved. A formal human-subject trust study remains outside the current Praxis unless separately approved and conducted.


In [ ]:
# Summary statistics for cautious narrative generation.
fidelity_top = fidelity_results.loc[fidelity_results["evidence_set"].eq("top_ranked")]
fidelity_random = fidelity_results.loc[fidelity_results["evidence_set"].eq("random_matched")]
mean_top_drop = float(fidelity_top["auprc_drop"].mean()) if not fidelity_top.empty else float("nan")
mean_random_drop = float(fidelity_random["auprc_drop"].mean()) if not fidelity_random.empty else float("nan")
mean_top_probability_change = (
    float(fidelity_top["mean_absolute_probability_change"].mean()) if not fidelity_top.empty else float("nan")
)
mean_random_probability_change = (
    float(fidelity_random["mean_absolute_probability_change"].mean()) if not fidelity_random.empty else float("nan")
)
mean_top_positive_drop = (
    float(fidelity_top["mean_probability_drop_positive_events"].mean()) if not fidelity_top.empty else float("nan")
)
mean_random_positive_drop = (
    float(fidelity_random["mean_probability_drop_positive_events"].mean()) if not fidelity_random.empty else float("nan")
)
mean_stability_jaccard = float(stability_results["top_k_jaccard"].mean()) if not stability_results.empty else float("nan")
mean_stability_spearman = float(stability_results["spearman_rank_correlation"].mean()) if not stability_results.empty else float("nan")

latency_reference = {
    str(row.model): {
        "samples": require_int(row.samples, "latency samples"),
        "mean_ms": require_float(row.latency_mean_ms, "latency mean"),
        "p50_ms": require_float(row.latency_p50_ms, "latency p50"),
        "p95_ms": require_float(row.latency_p95_ms, "latency p95"),
    }
    for row in latency_summary.itertuples(index=False)
}

reference = {
    "model_version": MODEL_VERSION,
    "supporting_xai_question": (
        "How faithfully, consistently, traceably, and efficiently do feature-, temporal-, and relation-level "
        "explanations characterize the evidence contributing to candidate low-altitude event risk scores?"
    ),
    "hypothesis_policy": "Notebook 09 does not modify H1, H2, or H3.",
    "fidelity": {
        "primary_metric": "mean_absolute_probability_change",
        "mean_top_ranked_probability_change": mean_top_probability_change,
        "mean_random_matched_probability_change": mean_random_probability_change,
        "top_minus_random_probability_change": mean_top_probability_change - mean_random_probability_change,
        "mean_top_ranked_positive_probability_drop": mean_top_positive_drop,
        "mean_random_matched_positive_probability_drop": mean_random_positive_drop,
        "secondary_mean_top_ranked_auprc_drop": mean_top_drop,
        "secondary_mean_random_matched_auprc_drop": mean_random_drop,
    },
    "stability": {
        "mean_top_k_jaccard": mean_stability_jaccard,
        "mean_spearman_rank_correlation": mean_stability_spearman,
        "noise_std": NOISE_STD,
        "top_k": TOP_K,
    },
    "relation_consistency": {
        "spearman_norm_vs_auprc_drop": relation_spearman_auprc,
        "spearman_norm_vs_probability_change": relation_spearman_probability,
    },
    "provenance": {
        "audit_checks": len(provenance_audit),
        "passed_checks": int(provenance_audit["passed"].sum()),
        "pass_fraction": provenance_coverage,
    },
    "explanation_latency": latency_reference,
    "analyst_cases": len(case_studies),
    "interpretation_boundary": (
        "XAI outputs describe model influence. They do not establish causal linkage, real site coincidence, "
        "or proof that an aircraft generated the replayed network activity."
    ),
}
REFERENCE_OUT.write_text(json.dumps(reference, indent=2, default=str), encoding="utf-8")

configuration = {
    "model_version": MODEL_VERSION,
    "random_state": RANDOM_STATE,
    "device": str(DEVICE),
    "fast_mode": FAST_MODE,
    "top_k": TOP_K,
    "fidelity_random_repeats": FIDELITY_RANDOM_REPEATS,
    "stability_repeats": STABILITY_REPEATS,
    "noise_std": NOISE_STD,
    "transformer_fidelity_sample_max": TRANSFORMER_FIDELITY_SAMPLE_MAX,
    "transformer_stability_sample_max": TRANSFORMER_STABILITY_SAMPLE_MAX,
    "rgcn_attribution_sample_max": RGCN_ATTRIBUTION_SAMPLE_MAX,
    "case_study_count": CASE_STUDY_COUNT,
    "latency_sample_max": LATENCY_SAMPLE_MAX,
    "local_hops": LOCAL_HOPS,
}
CONFIG_OUT.write_text(json.dumps(configuration, indent=2, default=str), encoding="utf-8")

narrative_lines = [
    "# XAI Validation Results — Notebook 09",
    "",
    "Notebook 09 evaluates explainability as a technical property of the frozen Transformer and R-GCN rather than treating attribution plots alone as sufficient evidence.",
    "",
    f"Top-ranked evidence removal produced a mean absolute prediction change of {mean_top_probability_change:.6f}, compared with {mean_random_probability_change:.6f} for matched random feature removal. Signed AUPRC changes are retained as secondary diagnostics because removing influential evidence can improve or degrade ranking performance.",
    f"Across small transformed-input perturbations, the mean top-{TOP_K} explanation overlap was {mean_stability_jaccard:.3f}, with mean Spearman rank correlation {mean_stability_spearman:.3f}.",
    f"R-GCN relation-weight norms and relation-removal AUPRC effects had a Spearman correlation of {relation_spearman_auprc:.3f}; this is treated as a diagnostic consistency measure rather than causal evidence.",
    f"The provenance audit passed {int(provenance_audit['passed'].sum())} of {len(provenance_audit)} checks ({provenance_coverage:.1%}), preserving traceability from analyst cases to frozen point-event and flow identifiers.",
    "",
    "Explanation latency is reported separately from prediction latency because analyst-facing evidence generation adds computational work after or alongside scoring.",
    "",
    "## Interpretation boundary",
    "",
    "The aircraft and NF-UNSW-NB15-v3 sources are independent historical datasets connected through controlled replay. Feature, temporal, flow, and relation attributions indicate model sensitivity or influence only; they do not demonstrate that the replayed network activity occurred at PANC, was generated by an observed aircraft, or caused the aircraft behavior.",
]
NARRATIVE_OUT.write_text("\n".join(narrative_lines) + "\n", encoding="utf-8")

display(pd.DataFrame([reference]))


In [ ]:
# Data dictionary, build summary, and checksums.
artifact_descriptions = {
    GLOBAL_SUMMARY_OUT.name: "Consolidated global Transformer and R-GCN feature, temporal, and relation diagnostics.",
    FIDELITY_OUT.name: "Top-ranked versus random feature-removal and R-GCN relation-removal fidelity tests.",
    STABILITY_OUT.name: "Explanation ranking stability under small transformed-input perturbations.",
    RELATION_CONSISTENCY_OUT.name: "R-GCN relation norms compared with behavioral relation-removal effects.",
    LATENCY_OUT.name: "Summary of analyst-explanation latency for Transformer and R-GCN.",
    LATENCY_SAMPLES_OUT.name: "Per-event explanation latency measurements.",
    PROVENANCE_OUT.name: "Traceability audit from XAI evidence back to frozen event and flow identifiers.",
    MODEL_MATRIX_OUT.name: "Comparison of explanation capabilities across tabular, Transformer, and R-GCN models.",
    CASE_STUDIES_OUT.name: "Representative held-out analyst cases with local feature, relation, and provenance evidence.",
    REFERENCE_OUT.name: "Frozen XAI validation reference metrics and interpretation boundary.",
    CONFIG_OUT.name: "Notebook 09 evaluation configuration.",
    NARRATIVE_OUT.name: "Cautious narrative summary for Chapter 4/5 drafting.",
}
data_dictionary = pd.DataFrame(
    [{"artifact": name, "artifact_role": description} for name, description in artifact_descriptions.items()]
)
data_dictionary.to_csv(DATA_DICTIONARY_OUT, index=False)

input_checksums = {
    str(path.relative_to(PROJECT_ROOT)): sha256_file(path)
    for path in required_paths
    if path.is_file()
}

summary = {
    "model_version": MODEL_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "inputs": {
        "point_events": len(model_data),
        "test_events": int(model_data[SPLIT_COLUMN].eq("test").sum()),
        "selected_flow_edges": len(selected_edges),
        "selected_flow_nodes": len(flow_table),
        "sha256": input_checksums,
    },
    "reproduction_checks": {
        "transformer_max_abs_probability_error": transformer_max_abs_error,
        "rgcn_max_abs_probability_error": rgcn_max_abs_error,
    },
    "outputs": {
        "global_summary_rows": len(global_summary),
        "fidelity_rows": len(fidelity_results),
        "stability_rows": len(stability_results),
        "relation_consistency_rows": len(relation_consistency),
        "latency_samples": len(latency_samples),
        "provenance_checks": len(provenance_audit),
        "analyst_cases": len(case_studies),
        "analyst_cards": len(list(ANALYST_CARDS_DIR.glob("*.md"))),
        "figures": len(list(FIGURES_DIR.glob("*.png"))),
    },
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "torch": torch.__version__,
        "device": str(DEVICE),
        "cuda_available": torch.cuda.is_available(),
    },
    "interpretation": reference["interpretation_boundary"],
}
SUMMARY_OUT.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

artifact_paths = [
    GLOBAL_SUMMARY_OUT,
    FIDELITY_OUT,
    STABILITY_OUT,
    RELATION_CONSISTENCY_OUT,
    LATENCY_OUT,
    LATENCY_SAMPLES_OUT,
    PROVENANCE_OUT,
    MODEL_MATRIX_OUT,
    CASE_STUDIES_OUT,
    REFERENCE_OUT,
    CONFIG_OUT,
    SUMMARY_OUT,
    NARRATIVE_OUT,
    DATA_DICTIONARY_OUT,
]
artifact_paths.extend(sorted(FIGURES_DIR.glob("*.png")))
artifact_paths.extend(sorted(ANALYST_CARDS_DIR.glob("*.md")))
checksum_lines: list[str] = []
for artifact_path in artifact_paths:
    if artifact_path.is_file():
        checksum_lines.append(
            f"{sha256_file(artifact_path)}  {artifact_path.relative_to(OUTPUT_DIR).as_posix()}"
        )
CHECKSUMS_OUT.write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")

print("Notebook 09 completed successfully.")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Fidelity rows:    {len(fidelity_results):,}")
print(f"Stability rows:   {len(stability_results):,}")
print(f"Analyst cases:    {len(case_studies):,}")
print(f"Provenance pass:  {provenance_coverage:.1%}")


## Notebook 09 output package

The completed notebook writes to:

```text
data/processed/point_event_v2/xai_leakage_safe_v2/
```

Principal artifacts:

- `xai_global_summary_v2.csv`
- `xai_fidelity_results_v2.csv`
- `xai_stability_results_v2.csv`
- `xai_relation_consistency_v2.csv`
- `xai_explanation_latency_v2.csv`
- `xai_provenance_audit_v2.csv`
- `xai_model_comparison_matrix_v2.csv`
- `xai_case_studies_v2.csv`
- `xai_reference_v2.json`
- `xai_results_narrative_v2.md`
- `analyst_cards/*.md`
- `figures/*.png`
- `SHA256SUMS.txt`

These artifacts support the XAI contribution without altering the frozen detection hypotheses. The appropriate conclusion is whether the explanation layer is **technically faithful, reasonably stable, traceable, and computationally usable**, not whether it proves causality or empirically improves human trust.
